# Pipeline scRNA-seq: scanpy + SCENIC

**Versión 1.7.27**

Corre en Google Colab y hace dos cosas: agrupa las células por tipo con scanpy, y con SCENIC averigua qué genes las controlan. Se puede correr sobre datos de ejemplo o sobre el estudio real de lupus, según lo que se elija en la celda 1.

| Modo | Datos | Qué hace |
|---|---|---|
| `"ejemplo"` | Demostración | Médula ósea humana, más SCENIC sobre el dataset `tiny` |
| `"zenodo"` | Lupus real | Datos de zenodo, figuras del artículo, SCENIC sobre células B |

Se generan las mismas 12 gráficas en los dos modos (violines de QC, scatter, HVG, PCA, grafo de vecinos, UMAP de QC, anotación, marcadores, DE, DE completo y top-100, heatmap, trayectoria), con scanpy y SCENIC aplicados de principio a fin.

Para usarlo: en la celda 1 se pone `MODO = "ejemplo"` o `"zenodo"`, y luego *Entorno de ejecución → Ejecutar todas*. Las celdas del otro modo se saltan solas.

La celda 1 trae `SALTAR_GRAFICAS = True`: se omiten las 12 gráficas y los cálculos que solo sirven para dibujarlas, y se llega a SCENIC en una fracción del tiempo. Es lo cómodo para comprobar que todo corre, pero **esa corrida no es un entregable**: para el análisis completo hay que ponerlo en `False`.

Dos cosas que conviene saber antes de correrlo.

En modo `"zenodo"`, por defecto se usan las 169.513 células completas del estudio (el número está confirmado en Zenodo y en la publicación de Jang), y eso necesita RAM alta: Colab Pro/Pro+, o el entorno de RAM alta activado. La celda 2.1 revisa cuánta RAM tiene la sesión y se detiene con una explicación si no alcanza. Si solo se tiene el Colab gratuito, se pone un número en `N_CELLS_MAX` (celda 1) para trabajar con una muestra en vez del dataset completo.

En modo `"ejemplo"` no se va a obtener biología real. El dataset tiene 500 genes y 20 factores de transcripción, muy poco para que SCENIC valide algún regulón. Aun así el notebook completa el flujo entero hasta mostrar un UMAP de regulones, marcado `_SIN_VALIDAR`. La celda 6.7, al final, indica si el resultado es apto para entregar (solo lo es en modo zenodo, con regulones validados).


---
# 1 · Configuración

Es la única celda que se necesita tocar. Se elige el modo y se ajustan los parámetros que se quieran cambiar.


In [ ]:
# ============================================================
#   PANEL DE CONTROL: todo lo configurable del analisis
# ============================================================
# Es la unica celda que hay que tocar para adaptar el analisis.
# El resto del notebook lee de aqui.

# Que datos analizar.
#   "ejemplo" = medula osea publica, para ver el flujo funcionando en minutos.
#   "zenodo"  = el dataset real del estudio, que se descarga de Zenodo.
MODO = "zenodo"           # "ejemplo"  o  "zenodo"

# ---- Modo rapido -------------------------------------------
# En True se saltan las 12 graficas y las dos figuras del estudio, junto con
# los calculos que solo sirven para dibujarlas (PCA, vecinos, UMAP, clustering,
# expresion diferencial y trayectoria). Sirve para comprobar en poco tiempo que
# SCENIC llega hasta el final. Una corrida asi no es entregable: le faltan las
# graficas y las tablas de expresion diferencial.
SALTAR_GRAFICAS = False

# ---- Cuantas celulas analizar (solo MODO="zenodo") ---------
# El dataset trae 169.513 celulas. Con 0 se procesan todas, lo que exige activar
# RAM alta en Colab (Entorno de ejecucion -> Cambiar tipo de entorno).
# Con un numero menor se toma una muestra estratificada por tipo celular, que
# conserva la proporcion de cada uno. El muestreo solo estratifica por esa
# columna: los grupos que forma la seccion 5.2, que cruza 9 pacientes con varios
# momentos, quedan mas cortos cuanto mas se recorte.
N_CELLS_MAX       = 0        # 0 = las 169.513 completas (requiere RAM alta)

# ---- Modo depuracion (solo MODO="zenodo") ------------------
# Leer el archivo del estudio (1,6 GB) tarda varios minutos cada vez. En True se
# prepara una vez una copia pequena y el resto del notebook trabaja sobre ella,
# lo que permite probar cambios en segundos. La copia sale de recortar el
# archivo original, asi que conserva su estructura real.
# Los resultados de una corrida en este modo son solo para probar, nunca para
# entregar.
RDS_DEBUG      = False
N_CELLS_DEBUG  = 4000                     # tamano de la copia de pruebas
RDS_DEBUG_FILE = 'RTX_zenodo_debug.RDS'

# ---- SCENIC: tamano del analisis ---------------------------
# SCENIC trabaja solo sobre las celulas B, y cuantas hay sale de N_CELLS_MAX:
# con 60.000 son unas 5.968, y con el dataset completo unas 16.861.
# Por defecto entran todas las que haya, de modo que este analisis crece cuando
# se amplia N_CELLS_MAX en vez de quedarse fijo.
# Poniendo SCENIC_DOWNSAMPLE en True se recortan a SCENIC_N_CELLS antes de
# reconstruir la red, que es el paso cuyo costo crece mas rapido con el numero
# de celulas. Sirve para pruebas rapidas, o si la corrida completa no cabe en el
# tiempo disponible.
SCENIC_DOWNSAMPLE = False    # True = recortar a SCENIC_N_CELLS
SCENIC_N_CELLS    = 2000     # solo se usa si SCENIC_DOWNSAMPLE = True
SCENIC_N_GENES    = 1500     # genes mas informativos que entran al analisis

# Algoritmo con el que se reconstruye la red.
METODO_GRN = "grnboost2"  # "grnboost2"     = el metodo oficial de SCENIC
                          # "sklearn_aprox" = aproximacion, solo para demostrar
# La red no se recorta a mano en ningun momento. SCENIC ya aplica sus propios
# umbrales al formar los grupos de genes, y filtrar antes se apartaria del
# procedimiento publicado.

# ---- Exigir que los regulones esten validados --------------
# SCENIC corre en tres pasos encadenados y este flag decide que pasa si el
# segundo no deja nada: reconstruir la red, filtrarla, y puntuar el resultado
# celula a celula. Es el filtro intermedio el que puede devolver una lista
# vacia.
# En False el notebook se detiene ahi, que es lo apropiado para un entregable.
# En True sigue al tercer paso y marca todas las salidas como no validadas.
# Con MODO="ejemplo" siempre sigue: ese dataset es demasiado pequeno para pasar
# el filtro y existe solo para recorrer el flujo entero.
PERMITIR_REGULONES_SIN_VALIDAR = False

# ---- Columnas del estudio (None = detectar sola) -----------
# El notebook busca por su cuenta que columna guarda cada dato. Si no la
# encuentra, muestra las columnas disponibles y se detiene para que se indique
# aqui cual usar.
COL_COND  = None    # diagnostico (lupus o control sano)
COL_TIME  = None    # momento del tratamiento
# El dataset trae dos columnas de tipo celular con distinto nivel de detalle. La
# primera agrupa todas las celulas B bajo una sola etiqueta, con lo que la figura
# de la seccion 5.1 saldria con una unica categoria. La segunda las desglosa en
# siete, que es lo que esa figura necesita.
COL_CTYPE = 'Celltype_level2'    # tipo celular anotado por los autores

# ---- Los dos grupos que compara la seccion 5.2 -------------
# Se derivan del nombre de cada muestra. Se pueden fijar aqui para comparar otro
# par de los disponibles.
PRE_LABEL  = None
POST_LABEL = None

# ---- Integridad de las descargas ---------------------------
# Comprueba que las bases de datos de SCENIC llegaron completas comparando su
# huella digital. Una descarga cortada produciria resultados silenciosamente
# equivocados.
VERIFICAR_CHECKSUMS = True

# ---- Trayectoria celular (grafica 12) ----------------------
# Grupo desde el que arranca el ordenamiento. Se escribe como texto, por ejemplo
# "0". Con None lo elige el propio notebook.
ROOT_CLUSTER      = None

# ---- Agrupamiento de referencia ----------------------------
# Los grupos de celulas que usan la anotacion, la expresion diferencial y la
# trayectoria. El numero es la resolucion: mas alto da mas grupos y mas finos.
CLUSTER_KEY       = "leiden_res_0.50"
# ============================================================
assert MODO in ("ejemplo", "zenodo"), f"MODO invalido: {MODO!r}"
assert METODO_GRN in ("grnboost2", "sklearn_aprox"), f"METODO_GRN invalido: {METODO_GRN!r}"
print("MODO:", MODO, "| GRN:", METODO_GRN)
if SALTAR_GRAFICAS:
    print("SALTAR_GRAFICAS=True: se omiten las 12 graficas, las figuras de la seccion 5\n"
          "       y los calculos que solo las alimentan. Se ejecutan QC, normalizacion\n"
          "       y la seccion 6 completa. Esta corrida no sirve como entregable.")
if METODO_GRN == "sklearn_aprox":
    print("AVISO: 'sklearn_aprox' NO es GRNBoost2. Los resultados no son comparables\n"
          "       con la literatura de SCENIC. Usalo solo para demostrar el flujo.")
if MODO == "zenodo" and RDS_DEBUG:
    print(f"RDS_DEBUG=True: se usara un RDS reducido de ~{N_CELLS_DEBUG:,} celulas.\n"
          "       Sirve para depurar rapido; los resultados NO son un entregable.")
if MODO == "zenodo" and N_CELLS_MAX == 0:
    print("N_CELLS_MAX=0: se usaran las 169.513 celulas completas del dataset.\n"
          "Esto requiere RAM alta; la celda siguiente lo verifica.")

---
# 2 · Preparar el entorno
Instala las librerías, igual para los dos modos. Si Colab pide reiniciar, hazlo y vuelve a *Ejecutar todas*.


### 2.1 · Memoria disponible

In [ ]:
import psutil, os

ram_gb = psutil.virtual_memory().total / 1e9
print(f"RAM: {ram_gb:.1f} GB | CPUs: {os.cpu_count()}")

# Aviso temprano de que la sesion se va a quedar corta de memoria. Los 20 GB son
# una referencia aproximada, no una medida del consumo real, asi que aqui solo
# se informa. La comprobacion que detiene el analisis esta mas adelante, junto
# al paso que de verdad consume la memoria, y con el valor de RAM leido en ese
# momento.
if MODO == "zenodo" and N_CELLS_MAX == 0 and ram_gb < 20:
    print(
        f"\nAVISO: pediste las 169.513 celulas completas (N_CELLS_MAX=0) y esta sesion\n"
        f"solo tiene {ram_gb:.1f} GB de RAM. La celda 3B.4 parara si se llega asi.\n"
        f"Activa RAM alta (Entorno de ejecucion -> Cambiar tipo de entorno de\n"
        f"ejecucion) o pon un numero en N_CELLS_MAX en la celda 1."
    )

### 2.2 · Instalar scanpy

In [ ]:
# La salida de pip se muestra entera. Si una instalacion falla conviene leer el
# motivo aqui y no descubrirlo a mitad del analisis.
import subprocess, sys

def pip_install(paquetes, etiqueta):
    """Instala y PARA si falla, mostrando el error real de pip."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *paquetes]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(
            f"Fallo la instalacion de {etiqueta} (codigo {r.returncode}).\n"
            "Revisa el error de pip arriba. NO sigas: las celdas siguientes\n"
            "fallarian de forma confusa o darian resultados incompletos."
        )
    print(f"{etiqueta}: instalado")

# Las versiones estan acotadas porque las librerias de analisis unicelular y las
# de calculo distribuido avanzan a ritmos distintos y no toda combinacion
# funciona. numpy queda entre 2.0 y 2.1: scanpy necesita la 2, y numba, que
# scanpy usa por dentro, todavia no admite la 2.1. Para dask se pide una version
# minima y ninguna maxima, y se pide junto a scanpy para que quede resuelto
# antes de que cualquier libreria lo cargue en memoria.
pip_install(["numpy>=2,<2.1", "dask>=2024.1", "distributed>=2024.1",
             "scanpy", "leidenalg", "igraph", "scikit-misc"], "scanpy + clustering")

# numpy 2 retiro una serie de nombres antiguos que algunas librerias siguen
# usando por dentro. Cuando falta uno, el error aparece en un sitio que no tiene
# nada que ver con la causa, asi que se reponen todos de una vez. Es un puente de
# compatibilidad que vive solo en memoria: no modifica ningun archivo instalado.
import warnings as _warn
import numpy as _np

# Lista escrita a mano. numpy publica su propio registro de nombres retirados,
# pero va vaciandolo con los anos, asi que no basta con consultarlo.
_ALIAS_EXTRA = {
    "unicode_": "str_", "string_": "bytes_", "round_": "round",
    "product": "prod", "cumproduct": "cumprod", "sometrue": "any",
    "alltrue": "all", "infty": "inf", "Inf": "inf", "Infinity": "inf",
    "NAN": "nan", "NaN": "nan", "float_": "float64", "complex_": "complex128",
    "mat": "asmatrix", "in1d": "isin", "row_stack": "vstack",
    "trapz": "trapezoid",
}

def _falta(nombre):
    """True si numpy ya no expone ese nombre (silencia el aviso de la sonda)."""
    with _warn.catch_warnings():
        _warn.simplefilter("ignore")
        return not hasattr(_np, nombre)

def _reponer_alias_numpy():
    """Repone en numpy los alias viejos. No toca ninguna libreria."""
    puestos = []
    def _poner(viejo, nuevo):
        if nuevo and _falta(viejo) and hasattr(_np, nuevo):
            setattr(_np, viejo, getattr(_np, nuevo))
            puestos.append(viejo)
    # Nombres que eran atajos a tipos basicos de Python.
    for viejo, tipo in (("object", object), ("bool", bool), ("int", int),
                        ("float", float), ("str", str), ("complex", complex)):
        if _falta(viejo):
            setattr(_np, viejo, tipo)
            puestos.append(viejo)
    for viejo, nuevo in _ALIAS_EXTRA.items():
        _poner(viejo, nuevo)
    # Para lo que numpy declare retirado y no este en la lista de arriba: su propio
    # mensaje de error suele nombrar el reemplazo, y de ahi se saca.
    for viejo, motivo in getattr(_np, "__expired_attributes__", {}).items():
        i = motivo.find("`np.")
        j = motivo.find("`", i + 1) if i != -1 else -1
        _poner(viejo, motivo[i + 4:j] if i != -1 and j != -1 else None)
    return puestos

print("Alias de numpy repuestos:", len(_reponer_alias_numpy()))

# Se comprueba que scanpy carga de verdad. Que pip termine bien no garantiza que
# la libreria arranque en este entorno.
try:
    import scanpy as sc, leidenalg, igraph
except (ImportError, AttributeError) as e:
    raise RuntimeError(
        f"No se pudo importar scanpy tras instalar: {type(e).__name__}: {e}\n\n"
        "Prueba: Entorno de ejecucion -> Desconectar y eliminar el tiempo de\n"
        "ejecucion, reconectar, y Ejecutar todas desde el principio."
    ) from e
print("scanpy:", sc.__version__, "| leidenalg:", leidenalg.version)

# Python no puede reemplazar una libreria que ya cargo en memoria. Si la sesion
# venia con una version antigua, la instalacion de arriba actualiza el disco pero
# el proceso sigue usando la vieja, y el fallo apareceria mucho despues. Se
# comprueba aqui, donde el mensaje todavia puede explicar que hacer.
import dask
try:
    _dv = tuple(int(p) for p in dask.__version__.split('.')[:2])
except ValueError:
    _dv = (9999, 0)          # version con formato raro: no bloqueamos por eso
if _dv < (2024, 1):
    raise RuntimeError(
        f"dask {dask.__version__} cargado en memoria; hace falta 2024.1 o mas nuevo.\n"
        "Se acaba de instalar uno moderno, pero Python no puede reemplazar en\n"
        "caliente un modulo ya cargado. Pasa en VMs donde corrio una version\n"
        "anterior de este notebook, que llegaba a fijar dask==2023.5.0.\n"
        "Solucion: Entorno de ejecucion -> Reiniciar sesion, y Ejecutar todas.\n"
        "Si aun asi persiste: Desconectar y eliminar el tiempo de ejecucion."
    )
print("dask:", dask.__version__)

### 2.3 · Instalar pySCENIC

In [ ]:
pip_install(["pyscenic==0.12.1"], "pySCENIC")

# Comprobacion de que las librerias de SCENIC no solo se instalaron, sino que
# cargan. pip puede terminar sin errores y aun asi dejar el entorno en un estado
# donde alguna no arranca, y entonces el fallo aparecería mucho mas adelante, ya
# empezado el analisis. Aqui se prueban todas de una vez.
faltan = []
for nombre in ("pyscenic", "ctxcore", "loompy", "arboreto", "dask", "distributed"):
    try:
        __import__(nombre)
    except Exception as e:
        faltan.append(f"  - {nombre}: {type(e).__name__}: {e}")

import pyscenic, numpy as np
print("pyscenic:", pyscenic.__version__, "| numpy:", np.__version__)

if faltan:
    print("\nNo se pudieron importar:\n" + "\n".join(faltan))
    raise RuntimeError(
        "Faltan dependencias de SCENIC.\n"
        "Causa habitual en Colab: pip instalo pyscenic pero el entorno necesita reinicio.\n"
        "Solucion: Entorno de ejecucion -> Reiniciar sesion -> Ejecutar todas."
    )
print("Dependencias de SCENIC: OK (incluye arboreto para GRNBoost2 real)")

### 2.4 · (Opcional) Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR='/content/drive/MyDrive/scanpy_scenic_lupus'; os.makedirs(PROJECT_DIR, exist_ok=True)
except Exception:
    PROJECT_DIR='.'
print("Resultados en:", PROJECT_DIR)


---
# 3 · Carga de datos

Produce el objeto `adata` (crudo, con conteos) según el modo. El análisis y las 12 gráficas de la sección 4 corren después sobre ese mismo `adata`, en los dos modos.


## 3A · [MODO EJEMPLO] Cargar médula ósea
> Solo corre si `MODO=="ejemplo"`.


In [ ]:
if MODO=="ejemplo":
    import scanpy as sc, anndata as ad, numpy as np, pandas as pd, pooch
    sc.settings.verbosity=1; sc.settings.set_figure_params(dpi=70, facecolor='white')
    np.random.seed(0)
    EX = pooch.create(path=pooch.os_cache('scverse_tutorials'),
                      base_url='doi:10.6084/m9.figshare.22716739.v1/')
    EX.load_registry_from_doi()
    samples={'s1d1':'s1d1_filtered_feature_bc_matrix.h5','s1d3':'s1d3_filtered_feature_bc_matrix.h5'}
    adatas={}
    for sid,fn in samples.items():
        a=sc.read_10x_h5(EX.fetch(fn)); a.var_names_make_unique(); adatas[sid]=a
    adata=ad.concat(adatas, label='sample'); adata.obs_names_make_unique()
    col_cond=col_time=col_ctype=None   # no aplican en ejemplo
    print("adata:", adata.shape)


## 3B · [MODO ZENODO] Descargar y convertir los datos de lupus
> Solo corre si `MODO=="zenodo"`. Conversión R→Python en proceso separado (ahorra RAM).


### 3B.1 · Descargar RDS de Zenodo (1.6 GB)

In [ ]:
if MODO == "zenodo":
    import os, hashlib, urllib.request, urllib.error

    RDS = 'RTX_zenodo.RDS'
    URL_RDS = "https://zenodo.org/records/17868028/files/RTX_zenodo.RDS?download=1"
    # Huella digital del archivo, si Zenodo la publica. Sirve para confirmar que la
    # descarga llego completa y sin alterar.
    SHA256_RDS = None

    def _es_rds(path):
        """saveRDS escribe gzip (1f 8b) o RDS sin comprimir ('RDX2'/'RDX3')."""
        with open(path, 'rb') as fh:
            m = fh.read(4)
        return m[:2] == b'\x1f\x8b' or m[:3] in (b'RDX', b'RDA')

    def _descargar_rds():
        parcial = RDS + '.part'
        print("Descargando RDS (~1.6 GB)...")
        try:
            with urllib.request.urlopen(URL_RDS, timeout=300) as resp:
                if resp.status != 200:
                    raise RuntimeError(f"HTTP {resp.status} desde Zenodo")
                declarado = resp.headers.get('Content-Length')
                declarado = int(declarado) if declarado else None
                leido = 0
                with open(parcial, 'wb') as fh:
                    while True:
                        trozo = resp.read(1 << 22)
                        if not trozo:
                            break
                        fh.write(trozo); leido += len(trozo)
                        if declarado and leido % (1 << 28) < (1 << 22):
                            print(f"  {leido/1e9:.2f} / {declarado/1e9:.2f} GB")
        except urllib.error.URLError as e:
            if os.path.exists(parcial):
                os.remove(parcial)
            raise RuntimeError(f"No se pudo descargar el RDS desde Zenodo:\n  {e}") from e

        real = os.path.getsize(parcial)
        if declarado is not None and real != declarado:
            os.remove(parcial)
            raise RuntimeError(
                f"Descarga truncada: {real:,} de {declarado:,} bytes. Re-ejecuta la celda."
            )
        if not _es_rds(parcial):
            os.remove(parcial)
            raise RuntimeError(
                "Lo descargado no es un archivo RDS (¿pagina de error de Zenodo?).\n"
                f"Comprueba a mano: {URL_RDS}"
            )
        os.replace(parcial, RDS)

    if not os.path.exists(RDS):
        _descargar_rds()
    elif not _es_rds(RDS):
        # Una descarga cortada de un intento anterior queda en disco con el nombre
        # correcto. Se comprueba el tamano para no arrastrar ese archivo a medias.
        print("El RDS local esta corrupto -> se descarga de nuevo")
        os.remove(RDS); _descargar_rds()

    if SHA256_RDS:
        h = hashlib.sha256()
        with open(RDS, 'rb') as fh:
            for blq in iter(lambda: fh.read(1 << 20), b''):
                h.update(blq)
        if h.hexdigest() != SHA256_RDS:
            raise RuntimeError(f"SHA256 del RDS no coincide:\n  esperado {SHA256_RDS}\n  obtenido {h.hexdigest()}")
        print("SHA256 del RDS: OK")

    print(f"RDS listo: {os.path.getsize(RDS)/1e9:.2f} GB (formato validado)")


### 3B.2 · Instalar SeuratObject en R

In [ ]:
if MODO == "zenodo":
    import subprocess, shutil

    if shutil.which('Rscript') is None:
        raise RuntimeError(
            "No hay R en este entorno. En Colab:  !apt-get install -y r-base\n"
            "El modo 'zenodo' necesita R para leer el objeto Seurat del RDS."
        )

    _r_code = (
        'for (p in c("SeuratObject","Matrix")) '
        'if (!requireNamespace(p, quietly=TRUE)) '
        'install.packages(p, repos="https://cloud.r-project.org"); '
        'ok <- all(sapply(c("SeuratObject","Matrix"), requireNamespace, quietly=TRUE)); '
        'if (!ok) quit(status=1); cat("paquetes R OK\\n")'
    )
    proc = subprocess.run(['Rscript', '-e', _r_code], capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        raise RuntimeError(
            "No se pudieron instalar SeuratObject/Matrix en R.\n"
            "Sin ellos la conversion del RDS (celda 3B.4) fallaria mas adelante."
        )


### 3B.2b · (Opcional) RDS reducido para pruebas

Solo hace algo si `RDS_DEBUG = True` en la celda 1. Genera **una vez** un RDS pequeño tomando una muestra estratificada del objeto real, y la celda 3B.4 lee de ahí en vez del de 1.6 GB.

Sirve para iterar rápido mientras se depura: la conversión deja de tardar minutos. La muestra se saca subconjuntando el objeto original, no reconstruyéndolo, así que conserva su estructura real —incluidas sus rarezas, como que la clave de célula viva en la columna `Barcode`— y las pruebas siguen siendo representativas.

El RDS reducido vive en el disco temporal de Colab: si se reinicia el entorno hay que regenerarlo (una vez por sesión, no por prueba).

Con `RDS_DEBUG = False`, que es el valor por defecto, esta celda no hace nada y el pipeline es exactamente el de entrega.


In [ ]:
if MODO == "zenodo" and RDS_DEBUG:
    import os, subprocess

    _R_DEBUG = r"""
suppressMessages({library(SeuratObject)})

fuente  <- Sys.getenv("RDS_FUENTE", "RTX_zenodo.RDS")
destino <- Sys.getenv("RDS_DESTINO", "RTX_zenodo_debug.RDS")
n_deb   <- as.integer(Sys.getenv("N_CELLS_DEBUG", "4000"))

obj <- readRDS(fuente)
meta <- obj@meta.data

# Los identificadores validos de celula son los de la matriz de expresion. La
# tabla de metadatos de este estudio esta numerada del 1 en adelante, sin
# nombres de celula, asi que consultarla para saber que celulas hay no sirve:
# devuelve numeros de fila que la matriz no reconoce.
ar <- if ("RNA" %in% Assays(obj)) "RNA" else DefaultAssay(obj)
celdas <- colnames(obj[[ar]])
if (is.null(celdas)) {
  cs0 <- tryCatch(GetAssayData(obj, assay=ar, layer="counts"),
                  error=function(e) tryCatch(GetAssayData(obj, assay=ar, slot="counts"),
                                             error=function(e2) NULL))
  if (is.null(cs0))
    stop("No se pudieron leer los nombres de celula del assay '", ar, "'.")
  celdas <- colnames(cs0); rm(cs0)
}
ncells <- length(celdas)
cat("Objeto completo:", ncells, "celulas\n")
cat("  celulas del assay ", ar, ", ejemplo: ", paste(head(celdas, 2), collapse=", "),
    "  | rownames(meta.data), ejemplo: ", paste(head(rownames(meta), 2), collapse=", "),
    "\n", sep="")
if (ncells != nrow(meta))
  stop("El assay tiene ", ncells, " celulas y meta.data ", nrow(meta), " filas.")

# Se busca que columna de los metadatos identifica de verdad a cada celula,
# comprobandola contra los nombres de la matriz de expresion.
clave <- rownames(meta)
usa_barcode <- FALSE
if (is.null(clave) || anyDuplicated(clave) || !setequal(clave, celdas)) {
  if (!("Barcode" %in% colnames(meta)))
    stop("rownames(meta.data) no identifican las celulas del assay y no hay una ",
         "columna 'Barcode' con la que alinear.")
  bc <- as.character(meta$Barcode)
  if (anyDuplicated(bc) || !setequal(bc, celdas))
    stop("Ni rownames(meta.data) ni la columna 'Barcode' identifican las celulas ",
         "del assay: no se puede muestrear con seguridad.")
  clave <- bc; usa_barcode <- TRUE
  cat("rownames(meta.data) no identifican celulas; se usa la columna 'Barcode'.\n")
}

# Se ponen los metadatos en el mismo orden que la matriz de expresion, y con los
# nombres de celula como identificador. Seurat recorta buscando esos nombres, de
# modo que sin este paso cualquier recorte devolveria un objeto vacio. El
# emparejamiento va por nombre, asi que cada fila conserva su propia celula.
obj@meta.data <- meta[match(celdas, clave), , drop=FALSE]
rownames(obj@meta.data) <- celdas

set.seed(0)
cc <- grep("celltype|cell_type|cell.type|annotation|ident", colnames(obj@meta.data),
           ignore.case=TRUE, value=TRUE)
if (length(cc) > 0) {
  # La muestra se toma por separado dentro de cada tipo celular, para que
  # conserve la proporcion original y no pierda las poblaciones minoritarias.
  # Al azar sobre el total, tipos escasos como HSPC o ILC podrian desaparecer.
  grp <- as.character(obj@meta.data[[cc[1]]]); fr <- n_deb / ncells
  # Caso aparte para los tipos celulares con una sola celula: la funcion de
  # muestreo de R interpreta un unico valor como un rango y devolveria una
  # celula distinta de la pedida.
  idx <- sort(unlist(lapply(split(seq_len(ncells), grp), function(ix)
    if (length(ix) == 1) ix else sample(ix, max(2, round(length(ix) * fr))))))
  cat("Muestreo estratificado por '", cc[1], "'\n", sep="")
} else {
  idx <- sort(sample(ncells, min(n_deb, ncells)))
  cat("Muestreo aleatorio\n")
}

# El recorte se hace sobre el objeto original en vez de construir uno nuevo, para
# que la copia de pruebas conserve la estructura del estudio tal como viene.
celulas <- celdas[idx]
pequeno <- tryCatch(subset(obj, cells = celulas), error = function(e)
  stop("No se pudo subconjuntar el objeto Seurat (", conditionMessage(e), ").\n",
       "Vuelve a RDS_DEBUG = False y trabaja con el RDS completo."))
rm(obj); gc()

# Los metadatos vuelven a quedar como venian en el archivo original: numerados
# por fila y con el identificador de celula solo en su columna. Una copia de
# pruebas mas ordenada que el estudio real no serviria para lo que existe, que es
# comprobar que el analisis maneja bien los datos tal como llegan.
if (usa_barcode) {
  ms <- pequeno@meta.data
  ms <- ms[order(match(rownames(ms), clave)), , drop=FALSE]
  rownames(ms) <- as.character(seq_len(nrow(ms)))
  pequeno@meta.data <- ms
  cat("Fixture: meta.data con rownames numericos y clave real en 'Barcode',\n",
      "         igual que el RDS completo.\n", sep="")
}

# Comprobaciones de que la copia mantiene lo que importa del original: la matriz
# de expresion, una fila de metadatos por celula y sus identificadores.
cs <- tryCatch(GetAssayData(pequeno, assay=ar, layer="counts"),
               error=function(e) tryCatch(GetAssayData(pequeno, assay=ar, slot="counts"),
                                          error=function(e2) NULL))
ms <- pequeno@meta.data
if (is.null(cs)) stop("El subconjunto no conserva la capa 'counts'.")
if (ncol(cs) != nrow(ms))
  stop("El subconjunto quedo descuadrado: ", ncol(cs), " celulas en counts y ",
       nrow(ms), " filas de metadatos.")
if ("Barcode" %in% colnames(ms) && !setequal(colnames(cs), as.character(ms$Barcode)))
  stop("El subconjunto perdio la correspondencia con la columna 'Barcode': ",
       "no reproduciria el comportamiento del RDS completo.")

# Se guarda sin comprimir. Ocupa mas espacio, pero se abre mucho mas rapido, que
# es justo lo que se busca en una copia pensada para repetir pruebas.
saveRDS(pequeno, destino, compress=FALSE)
cat("\nRDS reducido:", destino, "|", ncol(cs), "celulas x", nrow(cs), "genes\n")
cat("Tamano:", round(file.size(destino)/1e6), "MB (sin comprimir, carga rapido)\n")
"""

    if os.path.exists(RDS_DEBUG_FILE):
        print(f"Ya existe {RDS_DEBUG_FILE} ({os.path.getsize(RDS_DEBUG_FILE)/1e6:.0f} MB): se reutiliza.")
        print("Borralo a mano si quieres regenerarlo con otro N_CELLS_DEBUG.")
    else:
        with open('hacer_rds_debug.R', 'w') as fh:
            fh.write(_R_DEBUG)
        print(f"Generando {RDS_DEBUG_FILE} (~{N_CELLS_DEBUG:,} celulas). "
              "Esto lee el RDS completo UNA vez; las pruebas siguientes ya no.")
        entorno = dict(os.environ,
                       RDS_FUENTE=RDS,
                       RDS_DESTINO=RDS_DEBUG_FILE,
                       N_CELLS_DEBUG=str(N_CELLS_DEBUG))
        proc = subprocess.run(['Rscript', 'hacer_rds_debug.R'],
                              env=entorno, capture_output=True, text=True)
        print(proc.stdout)
        if proc.stderr.strip():
            print("--- stderr de R ---"); print(proc.stderr[-3000:])
        if proc.returncode != 0:
            raise RuntimeError(
                f"No se pudo generar el RDS reducido (codigo {proc.returncode}).\n"
                "Pon RDS_DEBUG = False en la celda 1 para trabajar con el RDS completo."
            )
elif MODO == "zenodo":
    print("RDS_DEBUG=False: la conversion usara el RDS completo (configuracion de entrega).")

### 3B.3 · Escribir el script de conversión

In [ ]:
%%writefile convert_rds.R
# El estudio se publica en el formato de Seurat, que es una herramienta de R,
# mientras que este analisis corre en Python con scanpy. Este script traduce de
# uno a otro: exporta la matriz de expresion, los metadatos de cada celula y las
# coordenadas del UMAP a formatos que Python lee.
# Todo lo que el script da por supuesto de la estructura del archivo se
# comprueba antes de usarlo, de modo que un archivo organizado de otra forma se
# detenga con un mensaje que diga que falta, en vez de exportar datos mal.
suppressMessages({library(SeuratObject); library(Matrix)})
n_max <- as.integer(Sys.getenv("N_CELLS_MAX", "60000"))

# El archivo a convertir se recibe desde fuera, para poder apuntar a la copia de
# pruebas sin tocar nada de este script.
fuente <- Sys.getenv("RDS_FILE", "RTX_zenodo.RDS")
cat("Leyendo:", fuente, "\n")
obj <- readRDS(fuente)
if (!inherits(obj, "Seurat"))
  stop("El RDS no contiene un objeto Seurat, sino: ", paste(class(obj), collapse="/"))

cat("=== ESTRUCTURA ===\n"); print(obj)
cat("\n=== COLUMNAS METADATA ===\n"); print(colnames(obj@meta.data))
meta <- obj@meta.data
cat("\n=== GRUPOS ===\n")
for (col in colnames(meta)) {
  v <- meta[[col]]
  if (is.factor(v) || is.character(v) || (is.numeric(v) && length(unique(v)) < 30))
    if (length(unique(v)) <= 30) { cat("\n[", col, "]\n", sep=""); print(table(v)) }
}

# --- La matriz de expresion --------------------------------------------------
# Se lee antes de decidir la muestra, porque el orden de sus columnas es el orden
# real de las celulas y es contra el que se alinea todo lo demas.
ar <- if ("RNA" %in% Assays(obj)) "RNA" else DefaultAssay(obj)
cat("Assay usado:", ar, "de", paste(Assays(obj), collapse=", "), "\n")

# El archivo puede traer varias matrices bajo distintos nombres, y el nombre no
# garantiza el contenido: en este dataset la que se llama de conteos ya viene
# transformada.
# Por eso la eleccion se hace mirando los valores y no el nombre. Los conteos sin
# procesar son enteros; cualquier version transformada tiene decimales. Si
# ninguna resulta de enteros se toma la primera y se avisa, y mas adelante el
# notebook mide en que estado llega la matriz y aplica solo lo que falte.
#
# La lectura va directa a la estructura interna del archivo en lugar de usar las
# funciones habituales, y solo trae un bloque de columnas de cada matriz. Esas
# funciones cargan la matriz entera, y con 169.513 columnas eso agota la memoria
# de la sesion antes de poder decidir nada.
assay_obj <- obj@assays[[ar]]

capas <- tryCatch(names(assay_obj@layers), error=function(e) NULL)
if (is.null(capas) || length(capas) == 0) capas <- c("counts", "data")
cat("Capas del assay:", paste(capas, collapse=", "), "\n")

leer_bloque <- function(nombre, n_celulas=2000L) {
  m <- tryCatch(assay_obj@layers[[nombre]], error=function(e) NULL)
  if (is.null(m))
    m <- tryCatch(slot(assay_obj, nombre), error=function(e) NULL)   # Assay clasico (pre-v5)
  if (is.null(m) || is.null(dim(m))) return(NULL)
  m[, seq_len(min(n_celulas, ncol(m))), drop=FALSE]
}
es_entera <- function(bloque) {
  if (is.null(bloque) || nrow(bloque) == 0 || ncol(bloque) == 0) return(NA)
  v <- if (inherits(bloque, "sparseMatrix")) bloque@x else as.numeric(bloque)
  if (length(v) == 0) return(NA)
  v <- v[seq_len(min(200000L, length(v)))]   # muestra: basta para decidir
  all(abs(v - round(v)) < 1e-8)
}

capa_usada <- NA_character_
for (cp in capas) {
  bloque <- leer_bloque(cp)
  ent <- es_entera(bloque)
  cat("  capa '", cp, "': ",
      if (is.na(ent)) "no legible"
      else if (ent) "valores ENTEROS (conteos crudos)"
      else "valores decimales (ya transformada)", "\n", sep="")
  rm(bloque); gc()
  # Se para en la primera que resulte de enteros, para no cargar las demas.
  if (!is.na(ent) && ent) { capa_usada <- cp; break }
}

if (is.na(capa_usada)) {
  capa_usada <- "counts"
  cat("AVISO: ninguna capa del assay trae conteos crudos (enteros).\n",
      "       Se usa 'counts' tal cual viene, ya normalizada en origen.\n",
      "       La celda 4.3 lo detecta y NO vuelve a normalizar.\n", sep="")
} else {
  cat("Capa elegida para la matriz: ", capa_usada, "\n", sep="")
}

# Ya elegida, la matriz se lee entera y una sola vez, por la misma via directa
# que la deteccion. Las funciones habituales de Seurat quedan como alternativa
# para archivos de otras versiones, donde esa via no existe.
counts <- tryCatch(assay_obj@layers[[capa_usada]], error=function(e) NULL)
if (is.null(counts))
  counts <- tryCatch(slot(assay_obj, capa_usada), error=function(e) NULL)  # Assay clasico (pre-v5)
if (is.null(counts))
  counts <- tryCatch(GetAssayData(obj, assay=ar, slot=capa_usada),
                     error=function(e) tryCatch(GetAssayData(obj, assay=ar, layer=capa_usada),
                                                error=function(e2) NULL))
rm(assay_obj); gc()
if (is.null(counts) || nrow(counts) == 0 || ncol(counts) == 0)
  stop("No se pudo leer ninguna matriz de expresion del assay '", ar, "'.")

# --- Emparejar cada celula con su anotacion ----------------------------------
# La matriz de expresion y la tabla de metadatos son dos listas independientes, y
# nada garantiza que traigan las celulas en el mismo orden. Si se emparejan por
# posicion cuando los ordenes difieren, cada fila de metadatos acaba pegada a la
# celula equivocada y todas sus columnas quedan mal asignadas.
# El emparejamiento va por nombre de celula, y se verifica. Es un punto critico
# porque un cruce aqui no produce ningun error visible: el analisis termina bien
# y entrega resultados que parecen validos sin serlo.
if (is.null(colnames(counts)))
  stop("La matriz de conteos no trae nombres de columna (celulas).")
if (ncol(counts) != nrow(meta))
  stop("La matriz tiene ", ncol(counts), " celulas y los metadatos ", nrow(meta), " filas.")

# En este estudio la tabla de metadatos esta numerada del 1 en adelante, sin
# nombres de celula: el identificador real vive en una columna aparte. Se prueba
# primero la numeracion de filas, por si otro estudio si la trae util, y se pasa
# a esa columna cuando no cubre las celulas de la matriz.
clave <- rownames(meta)
origen_clave <- "rownames(meta.data)"
cubre <- !is.null(clave) && !anyDuplicated(clave) && all(colnames(counts) %in% clave)

if (!cubre && "Barcode" %in% colnames(meta)) {
  bc <- as.character(meta$Barcode)
  if (!anyDuplicated(bc) && all(colnames(counts) %in% bc)) {
    clave <- bc; origen_clave <- "la columna 'Barcode'"; cubre <- TRUE
    cat("AVISO: rownames(meta.data) no sirven para alinear (no son identificadores\n",
        "       de celula validos, p.ej.: ", paste(head(rownames(meta), 3), collapse=", "), ").\n",
        "       Se usa la columna 'Barcode' como clave real de celula.\n", sep="")
  }
}

if (!cubre) {
  ejemplo_rn <- if (is.null(rownames(meta))) character(0) else head(rownames(meta), 3)
  stop("No se encontro una clave que alinee los metadatos con la matriz.\n",
       "  rownames(meta.data), ejemplo: ", paste(ejemplo_rn, collapse=", "), "\n",
       "  colnames(counts), ejemplo: ", paste(head(colnames(counts), 3), collapse=", "), "\n",
       "Ni rownames(meta.data) ni la columna 'Barcode' (si existe) cubren todas\n",
       "las celulas de la matriz. Revisa a mano que columna trae el barcode real.")
}

rownames(meta) <- clave
if (identical(colnames(counts), rownames(meta))) {
  cat("Alineacion matriz/metadatos: OK (mismo orden, clave=", origen_clave, ")\n", sep="")
} else {
  cat("Los metadatos se reordenan por nombre de celula (clave=", origen_clave,
      ") para que cada celula lleve su propia anotacion.\n", sep="")
  meta <- meta[colnames(counts), , drop=FALSE]
}

ncells <- ncol(counts)
if (n_max > 0 && n_max < ncells) {
  set.seed(0)
  cc <- grep("celltype|cell_type|cell.type|annotation|ident", colnames(meta),
             ignore.case=TRUE, value=TRUE)
  if (length(cc) > 0) {
    # La muestra se toma dentro de cada tipo celular, de modo que conserve las
    # proporciones del estudio completo.
    grp <- as.character(meta[[cc[1]]]); fr <- n_max / ncells
    idx <- sort(unlist(lapply(split(seq_len(ncells), grp),
                              function(ix) sample(ix, max(1, round(length(ix) * fr))))))
    cat("\n>>> DOWNSAMPLE estratificado por '", cc[1], "': ", ncells, " -> ", length(idx), "\n", sep="")
  } else {
    idx <- sort(sample(ncells, n_max))
    cat("\n>>> DOWNSAMPLE aleatorio:", ncells, "->", length(idx), "\n")
  }
} else {
  idx <- seq_len(ncells); cat("\n>>> TODAS:", ncells, "\n")
}

# Solo se copia la matriz cuando el recorte deja fuera alguna celula. Copiarla
# para quedarse con todas duplicaria varios gigabytes sin cambiar nada, y es
# suficiente para agotar la memoria de la sesion.
if (length(idx) < ncells) counts <- counts[, idx]
meta <- meta[idx, , drop=FALSE]
cat("Matriz de conteos:", nrow(counts), "genes x", ncol(counts), "celulas\n")

# --- Las coordenadas que ya trae el archivo ----------------------------------
# El archivo incluye un UMAP ya calculado. Se exporta para que la seccion 4 pueda
# contrastarlo con el que calcula por su cuenta. Son dos columnas por celula, asi
# que no pesa. Como el resto de tablas del archivo trae su propio orden de filas,
# de modo que se recorta por nombre y no por posicion.
emb <- NULL; un <- NA_character_
reds <- Reductions(obj)
if (length(reds) == 0) {
  cat("AVISO: el objeto no tiene reducciones; no se exporta el UMAP de los autores.\n")
} else {
  con_umap <- reds[grepl("umap", tolower(reds))]
  un <- if (length(con_umap) > 0) con_umap[1] else reds[1]
  e_all <- Embeddings(obj, un)
  if (!all(colnames(counts) %in% rownames(e_all))) {
    cat("AVISO: la reduccion '", un, "' no cubre todas las celulas; no se exporta.\n", sep="")
  } else {
    emb <- e_all[colnames(counts), , drop=FALSE]
  }
  rm(e_all)
}

# Ultima comprobacion de que todo sigue emparejado. Detenerse aqui es preferible
# a escribir en disco unos datos cruzados que nada volveria a revisar.
stopifnot(identical(colnames(counts), rownames(meta)))
if (!is.null(emb)) stopifnot(identical(colnames(counts), rownames(emb)))

# El archivo original se descarta antes de escribir, no despues. Contiene otra
# copia de la matriz que este analisis no usa, y liberarla deja sitio para la
# escritura. La matriz elegida sobrevive porque se guarda aparte.
rm(obj); gc()

Matrix::writeMM(counts, "counts.mtx")
write.csv(data.frame(gene=rownames(counts)), "genes.csv", row.names=FALSE)
write.csv(data.frame(barcode=colnames(counts)), "barcodes.csv", row.names=FALSE)
write.csv(meta, "metadata.csv")
if (!is.null(emb)) {
  write.csv(emb, "umap.csv")
  cat("Embedding exportado:", un, "(", ncol(emb), "dimensiones )\n")
}

rm(counts, meta); gc()
cat("\nListo.\n")


### 3B.4 · Ejecutar la conversión (lee el output: ahí están los grupos)

In [ ]:
if MODO == "zenodo":
    import os, subprocess, psutil

    # La memoria disponible se vuelve a medir aqui, justo antes del paso que la
    # consume, y no se reutiliza lo que calculo una celda anterior. En un
    # notebook las celdas pueden ejecutarse en cualquier orden, asi que un valor
    # heredado puede venir de una configuracion que ya se cambio y dejar pasar
    # una corrida condenada a quedarse sin memoria.
    ram_gb = psutil.virtual_memory().total / 1e9
    # Con la copia de pruebas no hay riesgo de memoria y la comprobacion no aplica.
    if N_CELLS_MAX == 0 and ram_gb < 20 and not RDS_DEBUG:
        raise RuntimeError(
            f"Pediste las 169.513 celulas completas (N_CELLS_MAX=0) pero esta sesion\n"
            f"solo tiene {ram_gb:.1f} GB de RAM. R se queda sin memoria al leer el RDS\n"
            f"y el sistema lo mata con SIGKILL (returncode -9), sin traceback util.\n\n"
            f"Opciones:\n"
            f"  1) Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> RAM\n"
            f"     alta (o Colab Pro/Pro+), y vuelve a ejecutar todo.\n"
            f"  2) Pon un numero en N_CELLS_MAX (celda 1), p.ej. 60000. El downsample\n"
            f"     es estratificado por tipo celular, asi que conserva las proporciones\n"
            f"     de las poblaciones y sigue siendo valido para el analisis."
        )

    archivo_rds = RDS_DEBUG_FILE if RDS_DEBUG else RDS
    entorno = dict(os.environ, N_CELLS_MAX=str(N_CELLS_MAX), RDS_FILE=archivo_rds)
    proc = subprocess.run(['Rscript', 'convert_rds.R'],
                          env=entorno, capture_output=True, text=True)
    print(proc.stdout)
    if proc.stderr.strip():
        print("--- stderr de R ---"); print(proc.stderr[-4000:])

    # Si la conversion falla conviene detenerse aqui. Sin esta comprobacion el
    # error saldria en el paso siguiente, como un archivo que no aparece, y ese
    # mensaje no dice nada de lo que ocurrio en realidad.
    if proc.returncode != 0:
        # Este codigo concreto significa que el sistema operativo corto el proceso
        # por falta de memoria. No es un fallo del programa, y el mensaje de
        # error no explicara nada porque no llego a escribirse ninguno.
        if proc.returncode == -9:
            detalle = (
                "\nEl codigo -9 es SIGKILL: el sistema mato a R por quedarse sin\n"
                "memoria (OOM). NO es un problema de SeuratObject ni del RDS, y el\n"
                "stderr de arriba no va a explicarlo porque R no llego a reaccionar.\n"
                "Reduce N_CELLS_MAX en la celda 1 o usa un entorno con RAM alta."
            )
        else:
            detalle = (
                "\nCausas habituales: SeuratObject no se instalo bien en la celda 3B.2,\n"
                "o el RDS no tiene la estructura esperada. El stderr de arriba deberia\n"
                "distinguir entre las dos."
            )
        raise RuntimeError(f"convert_rds.R fallo (codigo {proc.returncode}).{detalle}")

    obligatorios = ['counts.mtx', 'genes.csv', 'barcodes.csv', 'metadata.csv']
    faltan = [f for f in obligatorios if not os.path.exists(f)]
    if faltan:
        raise RuntimeError(
            f"R termino sin error pero no genero: {faltan}\n"
            "Probablemente el objeto Seurat no tiene la estructura esperada\n"
            "(assay 'RNA' con slot 'counts')."
        )
    for f in obligatorios + ['umap.csv']:      # umap.csv es opcional
        if os.path.exists(f):
            print(f"  {f}: {os.path.getsize(f)/1e6:.1f} MB")
        else:
            print(f"  {f}: no generado (el objeto no traia reduccion UMAP; se calculara una nueva)")

    print("\nConversion completada. Lee arriba la seccion '=== GRUPOS ===':\n"
          "ahi estan los valores reales de condicion y timepoint de este dataset.")


### 3B.5 · Reconstruir `adata` (preservando el UMAP de los autores)

In [ ]:
if MODO == "zenodo":
    import os, scipy.io, gc, scanpy as sc, anndata as ad, pandas as pd, numpy as np
    sc.settings.verbosity = 1; sc.settings.set_figure_params(dpi=70, facecolor='white')
    X = scipy.io.mmread('counts.mtx').T.tocsr()
    genes = pd.read_csv('genes.csv')['gene'].astype(str).values
    bc = pd.read_csv('barcodes.csv')['barcode'].astype(str).values
    meta = pd.read_csv('metadata.csv', index_col=0)
    meta.index = meta.index.astype(str)

    # El emparejamiento entre celulas y anotaciones se comprueba, no se da por
    # hecho. Pegar los identificadores sobre los metadatos sin verificar que
    # coinciden dejaria a cada celula con la anotacion de otra, sin ningun aviso,
    # y el analisis entregaria resultados que parecen validos sin serlo.
    if len(meta) != len(bc):
        raise ValueError(f"metadata.csv tiene {len(meta):,} filas y barcodes.csv {len(bc):,}.")
    if not meta.index.is_unique:
        raise ValueError("metadata.csv tiene barcodes duplicados: no se puede alinear con seguridad.")
    if not (meta.index.values == bc).all():
        ausentes = set(bc) - set(meta.index)
        if ausentes:
            raise ValueError(
                f"{len(ausentes):,} barcodes de la matriz no estan en metadata.csv, "
                f"p.ej.: {sorted(ausentes)[:3]}"
            )
        print("AVISO: metadata.csv venia en otro orden que barcodes.csv; se realinea por barcode.")
        meta = meta.loc[bc]

    adata = ad.AnnData(X=X, obs=meta, var=pd.DataFrame(index=genes))
    adata.obs_names = bc

    if os.path.exists('umap.csv'):                     # el UMAP de los autores es opcional
        umap = pd.read_csv('umap.csv', index_col=0)
        umap.index = umap.index.astype(str)
        if not (umap.index.values == bc).all():        # mismo cuidado que con los metadatos
            if set(bc) - set(umap.index):
                raise ValueError("umap.csv no cubre todos los barcodes de la matriz.")
            print("AVISO: umap.csv venia en otro orden; se realinea por barcode.")
            umap = umap.loc[bc]
        adata.obsm['X_umap_authors'] = umap.values
        print(f"UMAP de los autores conservado: {umap.shape[1]} dimensiones")
        del umap
    else:
        print("Sin UMAP de los autores; la seccion 4 calculara uno nuevo.")
    del X, meta; gc.collect()

    def _resumen_columnas():
        """Lista las columnas de obs con sus valores, para poder elegir a mano."""
        lineas = []
        for c in adata.obs.columns:
            v = adata.obs[c]
            n = v.nunique(dropna=True)
            if n <= 12:
                lineas.append(f"    {c!r:38s} ({n:2d} valores) -> {sorted(map(str, v.dropna().unique()))}")
            else:
                lineas.append(f"    {c!r:38s} ({n:4d} valores, continuo/ID)")
        return "\n".join(lineas)

    def resolver_columna(etiqueta, override, claves, obligatoria):
        """Devuelve la columna elegida. Nunca 'None en silencio':
        si es obligatoria y no aparece, para y te ensena las columnas reales."""
        if override is not None:
            if override not in adata.obs.columns:
                raise KeyError(
                    f"La columna {override!r} que fijaste para '{etiqueta}' no existe.\n"
                    f"Columnas disponibles:\n{_resumen_columnas()}"
                )
            print(f"  {etiqueta:6s}: {override!r}  (fijada a mano)")
            return override

        candidatas = [c for c in adata.obs.columns if any(k in c.lower() for k in claves)]
        if len(candidatas) == 1:
            print(f"  {etiqueta:6s}: {candidatas[0]!r}  (autodetectada)")
            return candidatas[0]
        if len(candidatas) > 1:
            print(f"  {etiqueta:6s}: {candidatas[0]!r}  (autodetectada; habia varias "
                  f"{candidatas} -> si no es la correcta fijala en la celda 1)")
            return candidatas[0]

        msg = (f"No se encontro ninguna columna para '{etiqueta}'.\n"
               f"Este notebook se escribio para el RDS; otro dataset\n"
               f"puede nombrar sus columnas de otra forma.\n"
               f"Solucion: elige la columna correcta de esta lista y ponla en la celda 1\n"
               f"como COL_{etiqueta.upper()} = 'nombre_de_columna'.\n"
               f"Columnas disponibles:\n{_resumen_columnas()}")
        if obligatoria:
            raise KeyError(msg)
        print(f"  {etiqueta:6s}: NO encontrada (opcional)\n{msg}\n")
        return None

    print("Resolviendo columnas de metadatos:")
    col_cond  = resolver_columna('cond',  COL_COND,
                                 ['disease', 'condition', 'group', 'sle', 'status', 'diagnosis'],
                                 obligatoria=False)
    col_time  = resolver_columna('time',  COL_TIME,
                                 ['time', 'visit', 'day', 'week', 'treatment', 'point'],
                                 obligatoria=False)
    # El tipo celular es imprescindible: sin el no hay forma de saber que celulas
    # son B, y las dos figuras del estudio y todo SCENIC dependen de eso.
    col_ctype = resolver_columna('ctype', COL_CTYPE,
                                 ['celltype', 'cell_type', 'cell.type', 'annotation', 'ident'],
                                 obligatoria=True)
    print()
    print(adata)


### 3B.6 · Guardar h5ad y liberar disco

In [ ]:
if MODO=="zenodo":
    import os, gc
    adata.write_h5ad(f"{PROJECT_DIR}/lupus_rituximab.h5ad")
    if os.path.exists('counts.mtx'): os.remove('counts.mtx')
    gc.collect(); print("Guardado.")


---
# 4 · Análisis y las 12 gráficas

Este bloque corre en los dos modos sobre `adata` y genera las 12 gráficas.


### 4.0 · Preparar marcadores según el modo
Se filtra los genes que sí existen en los datos (evita errores si falta alguno).


In [ ]:
import scanpy as sc, numpy as np, pandas as pd

def present(md_dict):
    out={k:[g for g in v if g in adata.var_names] for k,v in md_dict.items()}
    return {k:v for k,v in out.items() if v}

if MODO=="ejemplo":
    MARKERS = present({
        'CD14+ Mono':['FCN1','CD14'],'CD16+ Mono':['TCF7L2','FCGR3A','LYN'],
        'cDC2':['CST3','COTL1','LYZ','CLEC10A','FCER1A'],
        'Erythroblast':['MKI67','HBA1','HBB'],'Proerythroblast':['CDK6','SYNGR1','HBM','GYPA'],
        'NK':['GNLY','NKG7','CD247','TYROBP','KLRG1'],
        'Naive CD20+ B':['MS4A1','IL4R','IGHD','FCRL1','IGHM'],
        'Plasma cells':['MZB1','HSP90B1','PRDM1','IGKC','JCHAIN'],
        'CD4+ T':['CD4','IL7R','TRBC2'],'CD8+ T':['CD8A','CD8B','GZMK','CCL5','GZMB'],
        'T naive':['LEF1','CCR7','TCF7'],'pDC':['IL3RA','COBLL1','TCF4'],
    })
else:
    MARKERS = present({
        'T CD4':['CD3D','CD4','IL7R'],'T CD8':['CD3D','CD8A','GZMK'],
        'B':['MS4A1','CD79A','CD79B'],'NK':['GNLY','NKG7','KLRD1'],
        'Mono':['CD14','LYZ','FCGR3A'],'DC':['FCER1A','CST3'],
        'Plasma':['MZB1','JCHAIN','IGHG1'],
    })
print("Marcadores:", list(MARKERS.keys()))


### 4.1 · [Gráfica 1] Violines de QC
Número de genes por célula, conteos totales y % de conteos mitocondriales.


In [ ]:
adata.var['mt']  =adata.var_names.str.startswith('MT-')
adata.var['ribo']=adata.var_names.str.startswith(('RPS','RPL'))
adata.var['hb']  =adata.var_names.str.contains('^HB[^(P)]')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo','hb'], inplace=True, log1p=True)
if not SALTAR_GRAFICAS:
    sc.pl.violin(adata, ['n_genes_by_counts','total_counts','pct_counts_mt'],
                 jitter=0.4, multi_panel=True)
else:
    print("[SALTAR_GRAFICAS] Grafica 1 (violines de QC): omitida. Las metricas de QC si se calcularon.")


### 4.1b · Diagnóstico B2/B3 (conteos crudos y filtro mitocondrial)

Solo en `MODO="zenodo"`: compara las métricas de QC calculadas por scanpy contra las que ya traía el metadata de Seurat, para detectar si `convert_rds.R` extrajo conteos crudos de verdad y si los genes `MT-` siguen en la matriz.

In [ ]:
if MODO == "zenodo":
    import numpy as np

    # Primera comprobacion: si la matriz trae conteos sin procesar, el total que
    # calcula scanpy tiene que coincidir con la columna nCount_RNA que ya venia
    # en los metadatos. Que no coincidan indica que la matriz llega transformada.
    if "nCount_RNA" in adata.obs.columns:
        corr = np.corrcoef(adata.obs["total_counts"], adata.obs["nCount_RNA"])[0, 1]
        print("B2 - correlacion total_counts (scanpy) vs nCount_RNA (Seurat):", corr)
        cols_b2 = [c for c in ["total_counts", "nCount_RNA",
                               "n_genes_by_counts", "nFeature_RNA"]
                   if c in adata.obs.columns]
        print(adata.obs[cols_b2].describe())
        if corr < 0.99:
            print("\nB2: total_counts no sigue a nCount_RNA, senal de que la matriz no\n"
                  "    son conteos crudos. En el RDS. es lo esperado: la\n"
                  "    capa 'counts' viene ya normalizada desde origen y la matriz cruda\n"
                  "    no se publico. NO hace falta tocar nada: la celda 3B.4 elige la\n"
                  "    capa con enteros si existe alguna, y la 4.3 mide el estado real de\n"
                  "    la matriz y aplica solo la normalizacion que falte.\n"
                  "    Con esta matriz, 'total_counts' NO es profundidad de secuenciacion;\n"
                  "    para eso esta nCount_RNA, que si la conserva.")
    else:
        print("B2: 'nCount_RNA' no esta en los metadatos; no se puede comparar.")

    # Segunda comprobacion: scanpy calcula el porcentaje mitocondrial buscando los
    # genes cuyo nombre empieza por MT-. Si esa busqueda no encuentra ninguno, la
    # metrica sale cero en todas las celulas sin que nada falle. Se cuenta cuantos
    # hay en la matriz para poder distinguir un cero real de una columna vacia.
    mt_directo = int(adata.var_names.str.startswith("MT-").sum())
    print("\nB3 - genes MT- en adata.var_names:", mt_directo)
    if "percent.mt" in adata.obs.columns:
        print(adata.obs[["pct_counts_mt", "percent.mt"]].describe())
    else:
        print("B3: 'percent.mt' no esta en los metadatos de Seurat.")
    if mt_directo == 0:
        print("\nB3: no hay genes MT- en la matriz, por eso pct_counts_mt sale 0 en todas\n"
              "    las celulas. Los autores ya los quitaron del objeto (legitimo), pero\n"
              "    entonces el filtro de QC por mitocondrial no esta filtrando nada.\n"
              "    La columna 'percent.mt' de Seurat si conserva el valor que tenian\n"
              "    antes de quitarlos, y es la que hay que usar para hablar de calidad\n"
              "    mitocondrial en este dataset.")

### 4.2 · [Gráfica 2] Scatter de QC coloreado
Conteos totales vs. genes detectados, color = % mitocondrial.


In [ ]:
if not SALTAR_GRAFICAS:
    sc.pl.scatter(adata, 'total_counts', 'n_genes_by_counts', color='pct_counts_mt')
else:
    print("[SALTAR_GRAFICAS] Grafica 2 (scatter de QC): omitida.")

# El estudio de lupus llega ya depurado por sus autores, asi que solo se aplica
# una limpieza suave de genes. Los datos de ejemplo vienen sin procesar y
# necesitan el control de calidad completo.
if MODO=="ejemplo":
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.filter_genes(adata, min_cells=3)
    if not SALTAR_GRAFICAS:
        # Anade la columna 'predicted_doublet' con el resultado de la deteccion.
        # Solo marca: no elimina filas, asi que la matriz que sigue es la misma.
        sc.pp.scrublet(adata, batch_key='sample', random_state=0)
        print("Dobletes:", int(adata.obs['predicted_doublet'].sum()))
    else:
        print("[SALTAR_GRAFICAS] Deteccion de dobletes (scrublet): omitida.")
else:
    sc.pp.filter_genes(adata, min_cells=3)   # limpieza suave de genes


### 4.3 · [Gráfica 3] Selección de características: normalizado vs no
Se mide en qué estado llega la matriz y se aplica **solo la normalización que le falte**, en vez de darla por cruda. El RDS. trae la capa `counts` ya normalizada en origen, y volver a normalizarla la transformaba dos veces, falseando los HVG, el PCA y el clustering. Luego se marcan los genes altamente variables (HVG).

In [ ]:
import numpy as np, scipy.sparse as sp

# El pipeline espera la matriz normalizada y en escala logaritmica, pero no todos
# los datasets llegan en el mismo estado: este viene con los dos pasos ya
# aplicados. Repetirlos transformaria la matriz dos veces y cambiaria la
# seleccion de genes, el PCA y el clustering que salen despues.
# Por eso el estado se mide aqui, sobre la propia matriz, y se aplica solo lo que
# falte. Se distinguen tres casos y hay un cuarto que detiene la ejecucion:
#   valores enteros            -> sin procesar: se normaliza y se aplica log1p
#   decimales, suma constante  -> normalizada sin log1p: solo log1p
#   igual pero tras deshacer   -> ya viene completa: no se toca
#   ninguno de los anteriores  -> estado desconocido: se para
# La decision sale de los datos y no de una variable de configuracion, asi que
# sigue valiendo con otro dataset.
_m = adata.X[:min(200, adata.n_obs)]          # muestra: la decision es exacta por celula
_vals = _m.data if sp.issparse(_m) else np.asarray(_m).ravel()
_vals = _vals[_vals != 0]
_entera = _vals.size > 0 and np.allclose(_vals, np.round(_vals))

def _sumas_por_celula(mat, deshacer_log):
    m = mat.copy()
    if sp.issparse(m):
        if deshacer_log: m.data = np.expm1(m.data)   # expm1(0)=0: no rompe la dispersion
        return np.asarray(m.sum(axis=1)).ravel()
    a = np.asarray(m, dtype=float)
    if deshacer_log: a = np.expm1(a)
    return a.sum(axis=1)

def _es_constante(s):
    """True si todas las celulas suman lo mismo: la huella de una normalizacion."""
    s = s[np.isfinite(s)]
    # Se exige que la media sea positiva. Un grupo de celulas todas a cero tiene
    # desviacion cero y pasaria por normalizado, saltandose el paso sin avisar.
    if s.size == 0 or float(np.mean(s)) <= 0:
        return False
    # El margen no es mas estrecho porque la limpieza de genes del paso anterior
    # descarta genes poco frecuentes, y con ellos se va una parte pequena de
    # cada celula. La suma deja de ser exacta por poco, y el umbral tiene que
    # admitir esa desviacion sin llegar a confundirla con una matriz que nunca
    # se normalizo, que se aparta cien veces mas.
    return float(np.std(s)) / float(np.mean(s)) < 1e-2

if _entera:
    _estado = "conteos crudos"
    adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata); sc.pp.log1p(adata)
    _hecho = "se normaliza (CP10K) y se aplica log1p"
elif _es_constante(_sumas_por_celula(_m, True)):
    # Los valores tienen decimales y, al deshacer el logaritmo, todas las celulas
    # suman lo mismo: la huella de una normalizacion ya aplicada.
    _estado = "ya normalizada y con log1p aplicado en origen"
    adata.layers['normalizada_en_origen'] = adata.X.copy()
    # scanpy deja constancia de haber aplicado el logaritmo, y otras funciones
    # suyas consultan ese registro mas adelante, entre ellas la de expresion
    # diferencial. Como aqui el logaritmo venia de origen, la anotacion se
    # escribe a mano para que esas funciones encuentren lo que esperan.
    adata.uns.setdefault('log1p', {'base': None})
    _hecho = "NO se toca: ya viene normalizada y logaritmica"
elif _es_constante(_sumas_por_celula(_m, False)):
    _estado = "normalizada en origen, pero sin log1p"
    adata.layers['normalizada_en_origen'] = adata.X.copy()
    sc.pp.log1p(adata)
    _hecho = "solo se aplica log1p"
else:
    raise RuntimeError(
        "No se pudo determinar como viene la matriz de expresion.\n"
        "No son enteros (luego no son conteos crudos) y las celulas tampoco suman\n"
        "lo mismo, ni en crudo ni deshaciendo el log, asi que no es una\n"
        "normalizacion estandar. Puede estar escalada (z-score) o venir de un\n"
        "pipeline propio.\n"
        "Se para aqui a proposito: normalizar por encima de una matriz en un estado\n"
        "desconocido produce HVG, PCA y clusters que parecen resultados y no lo son.\n"
        "Revisa la salida de la celda 3B.4: ahi se lista capa por capa cual del\n"
        "assay trae enteros."
    )

print(f"Matriz de entrada: {_estado}")
print(f"  -> {_hecho}")

batch = 'sample' if 'sample' in adata.obs.columns else None
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key=batch)
if not SALTAR_GRAFICAS:
    sc.pl.highly_variable_genes(adata)   # muestra normalizado vs no
else:
    print("[SALTAR_GRAFICAS] Grafica 3 (HVG): omitida. La normalizacion y los HVG si se calcularon.")

### 4.4 · [Gráfica 4] PCA: PC1/PC2 y PC3/PC4
Coloreado por grupo y por % mitocondrial. Incluye la varianza explicada por componente.


In [ ]:
if not SALTAR_GRAFICAS:
    sc.tl.pca(adata, random_state=0)
    color_group = 'sample' if 'sample' in adata.obs.columns else (col_cond or 'pct_counts_mt')
    # Cada color se empareja por posicion con un par de componentes, asi que la
    # lista se repite para dibujar los mismos datos sobre PC1/PC2 y PC3/PC4.
    sc.pl.pca(adata,
              color=[color_group, color_group, 'pct_counts_mt', 'pct_counts_mt'],
              dimensions=[(0,1), (2,3), (0,1), (2,3)], ncols=2, size=3)
    sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
else:
    print("[SALTAR_GRAFICAS] Grafica 4 (PCA): omitida.")


### 4.5 · [Gráfica 5] Grafo de vecinos más cercanos
Construimos el grafo kNN y el UMAP; dibujamos las aristas del grafo sobre el UMAP.


In [ ]:
if not SALTAR_GRAFICAS:
    sc.pp.neighbors(adata, random_state=0)
    sc.tl.umap(adata, random_state=0)
    sc.pl.umap(adata, color=color_group, edges=True, edges_width=0.05,
               title='Grafo de vecinos mas cercanos')
else:
    print("[SALTAR_GRAFICAS] Grafica 5 (grafo de vecinos + UMAP): omitida.")


### 4.6 · Clustering Leiden (base para el resto)
Tres resoluciones. El resto de gráficas usa `CLUSTER_KEY` (por defecto res 0.50).


In [ ]:
if not SALTAR_GRAFICAS:
    for res in [0.02, 0.5, 2.0]:
        sc.tl.leiden(adata, key_added=f'leiden_res_{res:4.2f}', resolution=res,
                     flavor='igraph', n_iterations=2, random_state=0)
        print(f"res {res}: {adata.obs[f'leiden_res_{res:4.2f}'].nunique()} clusters")
else:
    print("[SALTAR_GRAFICAS] Clustering Leiden (solo alimenta graficas 6-12): omitida.")


### 4.7 · [Gráfica 6] Filtrado visto con UMAP (métricas de QC)
Las métricas de calidad proyectadas sobre el UMAP, para ver si algún cluster es artefacto.


In [ ]:
if not SALTAR_GRAFICAS:
    qc_cols=['n_genes_by_counts','total_counts','pct_counts_mt']
    if 'doublet_score' in adata.obs.columns: qc_cols.append('doublet_score')
    sc.pl.umap(adata, color=qc_cols, ncols=2, size=3)
else:
    print("[SALTAR_GRAFICAS] Grafica 6 (UMAP de metricas de QC): omitida.")


### 4.8 · [Gráfica 7] Anotación manual: números en cada grupo
Clusters con su número encima. En zenodo se muestra también la anotación de los autores.


In [ ]:
if not SALTAR_GRAFICAS:
    sc.pl.umap(adata, color=CLUSTER_KEY, legend_loc='on data', title='Clusters (numeros)')

    if MODO=="ejemplo":
        cluster_to_celltype={'0':'Lymphocytes','1':'Monocytes','2':'Erythroid','3':'B Cells'}
        adata.obs['cell_type']=(adata.obs['leiden_res_0.02'].map(cluster_to_celltype)
                                .fillna('Unknown').astype('category'))
        sc.pl.umap(adata, color='cell_type', legend_loc='on data')
    elif col_ctype:
        sc.pl.umap(adata, color=col_ctype, legend_loc='right margin',
                   title='Anotacion de los autores')
else:
    print("[SALTAR_GRAFICAS] Grafica 7 (anotacion de clusters): omitida.")


### 4.9 · [Gráfica 8] Patrones de expresión de marcadores + dotplot
Los patrones se generan solo con el nombre del gen, el símbolo; no hace falta secuencia. scanpy busca el gen en la matriz y colorea según su expresión.


In [ ]:
if not SALTAR_GRAFICAS:
    # Cada punto resume dos cosas a la vez: el tamano indica en que fraccion de las
    # celulas del grupo se detecta el gen, y el color, cuanto se expresa.
    sc.pl.dotplot(adata, MARKERS, groupby=CLUSTER_KEY, standard_scale='var')

    # Los mismos genes sobre el mapa celular, para ver donde se concentra cada uno.
    genes_umap=[g for gs in MARKERS.values() for g in gs][:6]
    sc.pl.umap(adata, color=genes_umap, ncols=3, size=3)
else:
    print("[SALTAR_GRAFICAS] Grafica 8 (marcadores + dotplot): omitida.")


### 4.10 · Expresión diferencial (base para gráficas 9-11)
Test de Wilcoxon: genes que definen cada cluster.

Antes del test excluimos los genes housekeeping (mitocondriales `MT-`, ribosomales `RPS`/`RPL` y hemoglobina `HB`). Dominan el ranking como ruido técnico, apareciendo como "diferenciales" en casi todos los clusters sin ser marcadores biológicos útiles. El DE se calcula sobre `adata_de`, la copia sin esos genes; `adata` queda intacto.


In [ ]:
if not SALTAR_GRAFICAS:
    # Se descartan por prefijo de nombre varias familias de genes que aparecen en
    # todas las celulas con valores altos. Coparian las primeras posiciones del
    # ranking sin diferenciar un grupo de otro.
    hk = (adata.var_names.str.startswith(('MT-','RPS','RPL','MRPS','MRPL')) |
          adata.var_names.str.contains('^HB[^(P)]'))
    adata_de = adata[:, ~hk].copy()
    print(f"Genes housekeeping excluidos del DE: {int(hk.sum())} | genes usados: {adata_de.n_vars}")
    sc.tl.rank_genes_groups(adata_de, groupby=CLUSTER_KEY, method='wilcoxon')
    print("Expresion diferencial calculada sobre", CLUSTER_KEY)
else:
    print("[SALTAR_GRAFICAS] Expresion diferencial (base de las graficas 9-11): omitida.")


### 4.11 · [Gráfica 9] Dotplot de genes diferencialmente expresados

In [ ]:
if not SALTAR_GRAFICAS:
    sc.pl.rank_genes_groups_dotplot(adata_de, groupby=CLUSTER_KEY,
                                    standard_scale='var', n_genes=5)
else:
    print("[SALTAR_GRAFICAS] Grafica 9 (dotplot de DE): omitida.")


### 4.12 · [Gráfica 10] Expresión diferencial: tabla completa + top-100 + gráficas

Se exportan dos archivos, sacados de la misma tabla:

- `DE_completo_<modo>.csv`: todos los genes evaluados por clúster (ya sin housekeeping), con puntaje, cambio de expresión (`logfoldchanges`) y significancia estadística (`pvals_adj`). Nada se descarta aquí.
- `top100_DE_<modo>.csv`: los genes realmente significativos (`pvals_adj < 0.05`) de cada clúster, hasta un máximo de 100.

Rellenar hasta 100 con genes que no pasan el corte estadístico daría un listado con el mismo aspecto en todos los clústeres, aunque buena parte no signifique nada, casi imposible de distinguir a simple vista de los genes que sí importan. Filtrando primero por significancia y recortando después se cumple el pedido de entrega, hasta 100 genes por clúster, sin fingir que hay más señal biológica de la que en realidad hay.


In [ ]:
if not SALTAR_GRAFICAS:
    de_all = sc.get.rank_genes_groups_df(adata_de, group=None)
    de_all['significativo'] = de_all['pvals_adj'] < 0.05

    # Tabla completa con todos los genes evaluados en cada grupo, resulten
    # significativos o no. Es la referencia para cualquier consulta posterior,
    # porque no aplica ningun recorte.
    out_csv = f"{PROJECT_DIR}/DE_completo_{MODO}.csv"
    de_all.to_csv(out_csv, index=False)

    # Resumen con los genes mas caracteristicos de cada grupo. Cien es un limite,
    # no una cantidad a completar: primero se descarta lo que no alcanza
    # significancia estadistica y solo despues se recorta la lista. Un grupo con
    # pocos genes significativos aparece con pocos, y no se rellena con ruido.
    top100 = (de_all[de_all['significativo']]
              .groupby('group', group_keys=False)
              .head(100))
    out_csv_top100 = f"{PROJECT_DIR}/top100_DE_{MODO}.csv"
    top100.to_csv(out_csv_top100, index=False)

    n_sig_por_cluster = de_all.groupby('group')['significativo'].sum().astype(int)
    print(f"Guardado: {out_csv}  ({len(de_all):,} filas = todos los genes evaluados x "
          f"{de_all['group'].nunique()} clusters)")
    print(f"Guardado: {out_csv_top100}  ({len(top100):,} filas: hasta 100 genes "
          f"REALMENTE significativos por cluster, sin relleno)")
    print("\nGenes significativos (pvals_adj < 0.05) por cluster:")
    print(n_sig_por_cluster.to_string())

    incompletos = n_sig_por_cluster[(n_sig_por_cluster < 100) & (n_sig_por_cluster > 0)]
    if len(incompletos) > 0:
        print(f"\n{len(incompletos)} cluster(es) con MENOS de 100 genes significativos: "
              "top100_DE trae solo los que hay, no se rellena para llegar a 100.")
        print(incompletos.to_string())

    sin_significativos = n_sig_por_cluster[n_sig_por_cluster == 0]
    if len(sin_significativos) > 0:
        print(f"\nAVISO: {len(sin_significativos)} cluster(es) sin NINGUN gen significativo "
              f"(pvals_adj < 0.05): {list(sin_significativos.index)}")
        print(f"No apareceran en top100_DE_{MODO}.csv: no hay nada valido que listar ahi. "
              "Siguen presentes en DE_completo con su ranking real, por si quieres revisarlos.")

    # Muestra en pantalla los primeros genes de cada grupo.
    display(top100.groupby('group').head(5))

    # Los genes mas caracteristicos de cada grupo, ordenados por su puntuacion.
    sc.pl.rank_genes_groups(adata_de, n_genes=20, sharey=False)
else:
    print("[SALTAR_GRAFICAS] Grafica 10 (tablas DE_completo y top100_DE): omitida.")


### 4.13 · [Gráfica 11] Heatmap de expresión diferencial
Con pocos genes por cluster y ejes intercambiados (`swap_axes`) para que los nombres se lean.


In [ ]:
if not SALTAR_GRAFICAS:
    sc.pl.rank_genes_groups_heatmap(adata_de, n_genes=3, groupby=CLUSTER_KEY,
                                    standard_scale='var', show_gene_labels=True,
                                    swap_axes=True, figsize=(12,14))
else:
    print("[SALTAR_GRAFICAS] Grafica 11 (heatmap de DE): omitida.")


### 4.14 · [Gráfica 12] Trayectoria (PAGA + pseudotiempo)
Reconstruye linajes: PAGA conecta los clusters según su similitud, y el pseudotiempo (DPT) ordena las células a lo largo del linaje. Necesita una célula raíz (`ROOT_CLUSTER`); si se deja en `None`, se elige sola.


In [ ]:
if not SALTAR_GRAFICAS:
    # Mide que grupos celulares estan conectados entre si y con que fuerza, lo que
    # da una idea de que tipos celulares derivan de cuales.
    sc.tl.paga(adata, groups=CLUSTER_KEY)
    sc.pl.paga(adata, color=CLUSTER_KEY, title='PAGA: conexiones entre clusters')

    # El UMAP se recalcula tomando esas conexiones como posicion de partida, en vez
    # de la inicializacion por defecto, para que el dibujo respete la estructura
    # del grafo anterior.
    sc.tl.umap(adata, init_pos='paga', random_state=0)

    # El pseudotiempo ordena las celulas a lo largo del grafo y necesita un grupo de
    # partida. Se toma el de ROOT_CLUSTER si esta fijado en la celda 1; si no, se
    # elige por posicion en el primer componente principal.
    import numpy as np
    root = ROOT_CLUSTER if ROOT_CLUSTER is not None else adata.obs[CLUSTER_KEY].value_counts().index[-1]
    adata.uns['iroot'] = int(np.flatnonzero(adata.obs[CLUSTER_KEY]==str(root))[0])
    sc.tl.dpt(adata)
    print("Cluster raiz del pseudotiempo:", root)
    sc.pl.umap(adata, color=[CLUSTER_KEY,'dpt_pseudotime'], legend_loc='on data', size=3)
else:
    print("[SALTAR_GRAFICAS] Grafica 12 (PAGA + pseudotiempo): omitida.")


---
# 5 · [MODO ZENODO] Figuras del artículo de lupus
> Solo corre si `MODO=="zenodo"`. Usa la anotación de los autores.


### 5.1 · Figura 1: Subtipos de células B


In [ ]:
# Que cuenta como celula B se decide en esta sola funcion, que usan tanto la
# figura de subtipos como el analisis de SCENIC. Teniendo el criterio en un unico
# sitio, las dos partes trabajan siempre sobre el mismo conjunto de celulas.
import re

B_KW = ['naive b', 'transitional', 'memory b', 'switched', 'abc',
        'plasmablast', 'plasma', 'b cell', 'b_cell']

# Etiquetas que designan celulas B sin nombrar ningun subtipo, propias de las
# anotaciones poco detalladas. Se comparan como texto completo y no como
# fragmento: buscar una "b" suelta dentro de cada etiqueta daria por celulas B a
# casi todas las demas.
B_EXACTAS = {'b', 'b cells', 'bcell', 'bcells', 'b-cell', 'b-cells', 'b lymphocyte'}

# Nombres habituales de las columnas que guardan el tipo celular. Sirven para
# proponer una alternativa cuando la columna elegida no encuentra celulas B.
_CLAVES_CTYPE = ('celltype', 'cell_type', 'cell.type', 'annotation', 'ident')


def _marcar_B(etiquetas):
    """Mascara booleana de celulas B para una serie de etiquetas."""
    bajas = etiquetas.str.lower().str.strip()
    # Las palabras se tratan como texto literal, de modo que anadir a la lista una
    # etiqueta con puntos o signos no altere la busqueda.
    patron = '|'.join(re.escape(k) for k in B_KW)
    return bajas.str.contains(patron, regex=True) | bajas.isin(B_EXACTAS)


def seleccionar_celulas_B(adata, col_ctype, min_celulas=50):
    """Devuelve el subconjunto de celulas B segun la columna de tipo celular.

    Las palabras que busca son las del dataset actual. Con otra nomenclatura el
    filtro no encuentra nada, y en ese caso la funcion para y muestra las
    etiquetas reales en vez de devolver un objeto vacio.
    """
    etiquetas = adata.obs[col_ctype].astype(str)
    es_B = _marcar_B(etiquetas)
    n = int(es_B.sum())

    if n < min_celulas:
        # Cuando no aparecen celulas B, lo habitual es que la columna consultada no
        # sea la adecuada, no que falten palabras en la lista. Los estudios
        # suelen traer varios niveles de anotacion, y la busqueda automatica
        # puede haberse quedado con el menos detallado. Antes de rendirse se
        # prueban las demas columnas y se indica cual funciona.
        alternativas = []
        for c in adata.obs.columns:
            if c == col_ctype or not any(k in c.lower() for k in _CLAVES_CTYPE):
                continue
            try:
                m = int(_marcar_B(adata.obs[c].astype(str)).sum())
            except Exception:
                continue
            if m >= min_celulas:
                alternativas.append((c, m))

        if alternativas:
            sugerencia = (
                "\n\nLa columna elegida no parece ser la correcta. Estas SI encuentran celulas B:\n"
                + "\n".join(f"    COL_CTYPE = {c!r}   ->  {m:,} celulas B" for c, m in alternativas)
                + "\n\nPonla en la celda 1 y vuelve a ejecutar desde la celda 3B.5.\n"
                  "No hace falta repetir la conversion de R."
            )
        else:
            sugerencia = (
                "\n\nNinguna otra columna de anotacion encuentra celulas B tampoco, asi que\n"
                "el problema si esta en la nomenclatura: ajusta B_KW en esta celda."
            )

        raise ValueError(
            f"El filtro de celulas B encontro {n} celulas (minimo esperado: {min_celulas}).\n"
            f"Se buscaron estas palabras clave en {col_ctype!r}: {B_KW}\n"
            f"Etiquetas reales en {col_ctype!r}:\n"
            + "\n".join(f"    {e!r}: {c:,}" for e, c in etiquetas.value_counts().items())
            + sugerencia
        )

    sub = adata[es_B].copy()
    print(f"Celulas B: {sub.n_obs:,} de {adata.n_obs:,} ({100*sub.n_obs/adata.n_obs:.1f}%)")

    # Encontrar celulas B no garantiza que la anotacion distinga subtipos. Con una
    # etiqueta unica la seleccion es correcta, pero la figura sale con una sola
    # categoria y deja de mostrar lo que su titulo anuncia.
    n_subtipos = sub.obs[col_ctype].astype(str).nunique()
    if n_subtipos < 2:
        print(f"AVISO: {col_ctype!r} solo distingue {n_subtipos} etiqueta(s) dentro de las\n"
              "       celulas B, asi que la figura no mostrara subtipos reales. Si el dataset\n"
              "       tiene una anotacion mas fina (p.ej. un 'level2'), ponla en COL_CTYPE.")
    return sub


if MODO == "zenodo" and not SALTAR_GRAFICAS:
    adata_B = seleccionar_celulas_B(adata, col_ctype)
    print(adata_B.obs[col_ctype].value_counts())
    sc.pl.umap(adata_B, color=col_ctype, size=8, title='Subtipos de celulas B (Lupus)')
elif MODO == "zenodo":
    print("[SALTAR_GRAFICAS] Figura 1 (subtipos de celulas B): omitida. "
          "La celda 6.2 vuelve a seleccionar las celulas B por su cuenta.")


### 5.2 · Figura 2: Volcano pre vs post-rituximab

Las etiquetas de *timepoint* se detectan solas a partir de los valores reales de la columna. Si la detección es ambigua (por ejemplo, hay varios momentos post-tratamiento), la celda se detiene y muestra los valores disponibles para que elijas: se fijan en `PRE_LABEL` / `POST_LABEL`, en la celda 1, sin tocar el código.


In [ ]:
if MODO == "zenodo" and not SALTAR_GRAFICAS:
    CELLTYPE_SUBSET = None      # p.ej. 'Memory B' para restringir el volcano a un tipo celular

    # La columna de momento tiene mas de dos valores y esta figura compara solo dos,
    # asi que hay que elegir cual va contra el estado previo. Se toma el temprano
    # porque es el unico con los 9 pacientes; al tardio le faltan 3, repartidos
    # entre las dos categorias de la columna Responder. Cambiar este valor por el
    # tardio es valido, contando con esos 3 pacientes menos.
    POST_POR_DEFECTO = 'Early'

    # --- El momento va dentro del identificador de muestra ----------------------
    # No hay columna propia: el identificador combina sujeto y momento separados
    # por un guion bajo. Parte de las muestras no llevan sufijo, y se reconocen
    # por no tener separador. La expresion regular exige ese separador, asi que
    # esas quedan sin valor y el filtro posterior las descarta solo.
    if col_time is None:
        if 'sampleID' not in adata.obs.columns:
            raise RuntimeError(
                "No hay columna de timepoint ni 'sampleID' del que derivarla.\n"
                "Fija COL_TIME en la celda 1 con la columna correcta (la celda 3B.5 las listo)."
            )
        sid = adata.obs['sampleID'].astype(str)
        adata.obs['timepoint'] = sid.str.extract(r'^[^_]+_(.+)$', expand=False)

        derivados = adata.obs['timepoint'].dropna().unique()
        sin_momento = int(adata.obs['timepoint'].isna().sum())
        if len(derivados) == 0:
            raise RuntimeError(
                "Se intento derivar el timepoint de 'sampleID' pero ningun valor tiene\n"
                "el formato 'PACIENTE_Momento'. Valores reales:\n"
                + "\n".join(f"    {v!r}" for v in sorted(sid.unique())[:20])
                + "\n\nFija COL_TIME en la celda 1 con la columna correcta."
            )
        col_time = 'timepoint'
        print(f"Timepoint derivado de 'sampleID' -> columna {col_time!r}")
        print(f"  momentos encontrados: {sorted(derivados)}")
        print(f"  celulas sin momento (controles sanos): {sin_momento:,}\n")

    valores = adata.obs[col_time].astype(str)
    disponibles = sorted(valores.unique())
    conteos = valores.value_counts()
    _lista = "\n".join(f"    {v!r}: {conteos[v]:,} celulas" for v in disponibles)

    def _elegir(termino, claves, excluir):
        """Busca UN valor que encaje. Si hay 0 o mas de 1, para y muestra las opciones."""
        cand = [v for v in disponibles
                if any(k in v.lower() for k in claves)
                and not any(x in v.lower() for x in excluir)]
        if len(cand) == 1:
            return cand[0]
        motivo = "no encaja ninguno" if not cand else f"encajan varios: {cand}"
        raise RuntimeError(
            f"No se pudo determinar automaticamente el valor de '{termino}' ({motivo}).\n"
            f"Valores reales de {col_time!r}:\n{_lista}\n\n"
            f"Elige el que corresponda y ponlo en la celda 1:\n"
            f"    PRE_LABEL  = '...'   # antes del rituximab\n"
            f"    POST_LABEL = '...'   # despues del rituximab"
        )

    # Al buscar el momento posterior se descarta cualquier etiqueta que empiece por
    # "pre", para que una palabra como "pretratamiento" no pase por posterior.
    pre = PRE_LABEL if PRE_LABEL is not None else _elegir(
        'PRE', ['pre', 'baseline', 'before', 'screening'], ['post'])

    # Buscar la palabra "post" no sirve en este estudio, donde los momentos se
    # llaman temprano y tardio. Se usa el valor elegido mas arriba, y solo si no
    # existe se recurre a la busqueda, que se detendra mostrando los momentos
    # disponibles.
    if POST_LABEL is not None:
        post = POST_LABEL
    elif POST_POR_DEFECTO in disponibles:
        post = POST_POR_DEFECTO
    else:
        post = _elegir('POST', ['post', 'after', 'follow'], ['pre-', 'pretreat'])

    for etiqueta, valor in (('PRE_LABEL', pre), ('POST_LABEL', post)):
        if valor not in disponibles:
            raise ValueError(f"{etiqueta}={valor!r} no existe en {col_time!r}.\n"
                             f"Valores reales:\n{_lista}")
    PRE_LABEL, POST_LABEL = pre, post
    print(f"Comparacion: {POST_LABEL!r}  vs  {PRE_LABEL!r}  (columna {col_time!r})")

    ad_de = adata
    if CELLTYPE_SUBSET and col_ctype:
        ad_de = ad_de[ad_de.obs[col_ctype].astype(str) == CELLTYPE_SUBSET]
        print(f"Restringido a {CELLTYPE_SUBSET!r}: {ad_de.n_obs:,} celulas")
    ad_de = ad_de[ad_de.obs[col_time].astype(str).isin([PRE_LABEL, POST_LABEL])].copy()

    n_por_grupo = ad_de.obs[col_time].astype(str).value_counts()
    print(n_por_grupo)
    if n_por_grupo.min() < 30:
        raise ValueError(
            f"Uno de los grupos tiene solo {n_por_grupo.min()} celulas.\n"
            "Con tan pocas, el test de Wilcoxon no es fiable y el volcano no seria interpretable.\n"
            "Sube N_CELLS_MAX, o quita CELLTYPE_SUBSET, o elige otros timepoints."
        )

    # Se cuentan tambien los pacientes de cada grupo, no solo las celulas. Las
    # celulas de un mismo paciente no son observaciones independientes, asi que el
    # conteo de celulas por si solo no dice si la comparacion esta equilibrada.
    if 'patientID' in ad_de.obs.columns:
        pac = ad_de.obs.groupby(ad_de.obs[col_time].astype(str), observed=True)['patientID'].nunique()
        print("\nPacientes por grupo:")
        print(pac.to_string())
        if pac.nunique() > 1:
            solo = {g: set(ad_de.obs.loc[ad_de.obs[col_time].astype(str) == g, 'patientID'].unique())
                    for g in pac.index}
            faltan = solo[PRE_LABEL] - solo[POST_LABEL]
            print(f"\nAVISO: los grupos no tienen los mismos pacientes. En {POST_LABEL!r} faltan: "
                  f"{sorted(faltan) if faltan else 'ninguno'}.\n"
                  "       La comparacion deja de estar pareada y parte de la diferencia puede\n"
                  "       venir de que pacientes entran en cada lado, no del tratamiento.")

    sc.tl.rank_genes_groups(ad_de, groupby=col_time, groups=[POST_LABEL],
                            reference=PRE_LABEL, method='wilcoxon')
    de = sc.get.rank_genes_groups_df(ad_de, group=POST_LABEL)
    print(f"Genes evaluados: {len(de):,}")
elif MODO == "zenodo":
    print("[SALTAR_GRAFICAS] Figura 2 (volcano pre/post-rituximab): omitida.")


In [ ]:
if MODO=="zenodo" and not SALTAR_GRAFICAS:
    import matplotlib.pyplot as plt, numpy as np
    d=de.dropna(subset=['logfoldchanges','pvals_adj']).copy()
    d['nlp']=-np.log10(d['pvals_adj'].clip(lower=1e-300))
    up=(d['logfoldchanges']>1)&(d['pvals_adj']<0.05); dn=(d['logfoldchanges']<-1)&(d['pvals_adj']<0.05)
    plt.figure(figsize=(8,6))
    plt.scatter(d['logfoldchanges'],d['nlp'],s=6,c='lightgray')
    plt.scatter(d.loc[up,'logfoldchanges'],d.loc[up,'nlp'],s=8,c='#c0392b',label='Up')
    plt.scatter(d.loc[dn,'logfoldchanges'],d.loc[dn,'nlp'],s=8,c='#2e5f9a',label='Down')
    for _,r in d[up|dn].nlargest(15,'nlp').iterrows(): plt.text(r['logfoldchanges'],r['nlp'],r['names'],fontsize=7)
    plt.axvline(0,color='k',lw=.5); plt.axhline(-np.log10(0.05),color='k',ls='--',lw=.5)
    plt.xlabel(f'logFC ({POST_LABEL} - {PRE_LABEL})'); plt.ylabel('-log10 p'); plt.legend()
    plt.title('Volcano post vs pre-rituximab'); plt.tight_layout(); plt.show()


---
# 6 · Parte B: Redes regulatorias (SCENIC)

GRNBoost2 → cisTarget → AUCell. Corre en ambos modos; la matriz de entrada cambia (celda 6.2).


### 6.0 · Compatibilidad de pySCENIC con numpy moderno

pySCENIC 0.12.1, de 2022, usa `np.object`, `np.bool`, etc. Numpy eliminó esos alias en la versión 1.24, así que en un Colab actual pySCENIC puede caerse con `AttributeError: module 'numpy' has no attribute 'object'`.

Esta celda restaura los alias, que no son más que `object`, `bool`, `int`..., de dos formas: en memoria para este proceso, y mediante un archivo `.pth` para que también los tenga el subproceso que lanza `pyscenic ctx`.

Esto evita que el proceso se caiga, pero no hace que aparezcan regulones: son dos problemas distintos. Si cisTarget termina bien pero devuelve 0 regulones, la causa es biológica o del dataset (pocos genes, TFs sin motivos en la base hg38), no este alias. La celda 6.4 distingue ambos casos.



In [ ]:
# SCENIC se publico cuando numpy usaba unos nombres que la version actual ya
# retiro. Aqui se reponen esos nombres para que la libreria funcione sin
# modificar ninguno de sus archivos ni instalar una version antigua de numpy,
# que romperia el resto del entorno.
import numpy as np, site, os, sys, subprocess, importlib

# Hace falta en dos sitios: en esta sesion y en el programa aparte que SCENIC
# lanza mas adelante, que arranca de cero y no hereda nada de lo que hay aqui.
# Se guarda como archivo para que las dos partes usen exactamente lo mismo.
_MODULO = '''"""Repone en numpy los alias que las versiones 1.24 y 2.0 retiraron.

Lo importa un archivo .pth, asi que corre al arrancar CUALQUIER interprete de
este entorno: por eso no imprime nada ni deja escapar avisos.
No modifica archivos de numpy ni de pySCENIC.
"""
import warnings

try:
    import numpy
except ImportError:
    # Si esto se ejecuta antes de que numpy este disponible, no hay nada que
    # reponer. Se sale en silencio en lugar de lanzar un error, que aparecería
    # en la salida de cualquier programa del entorno sin venir a cuento.
    numpy = None

# Nombres que eran atajos a tipos basicos de Python.
_TIPOS_124 = (("object", object), ("bool", bool), ("int", int),
              ("float", float), ("str", str), ("complex", complex))

# Lista escrita a mano. numpy publica su propio registro de nombres retirados,
# pero lo va vaciando con los anos, asi que consultarlo no basta.
_EXTRA = {"unicode_": "str_", "string_": "bytes_", "round_": "round",
          "product": "prod", "cumproduct": "cumprod", "sometrue": "any",
          "alltrue": "all", "infty": "inf", "Inf": "inf", "Infinity": "inf",
          "NAN": "nan", "NaN": "nan", "float_": "float64",
          "complex_": "complex128", "mat": "asmatrix", "in1d": "isin",
          "row_stack": "vstack", "trapz": "trapezoid"}


def _falta(nombre):
    """True si numpy ya no expone ese nombre (silencia el aviso de la sonda)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return not hasattr(numpy, nombre)


def _reemplazo(motivo):
    """De un mensaje "Use `np.str_` instead." saca "str_"."""
    i = motivo.find("`np.")
    if i == -1:
        return None
    j = motivo.find("`", i + 1)
    return motivo[i + 4:j] if j != -1 else None


def restaurar():
    """Repone los alias que falten y devuelve los nombres que hubo que reponer."""
    if numpy is None:
        return []
    puestos = []
    for viejo, tipo in _TIPOS_124:
        if _falta(viejo):
            setattr(numpy, viejo, tipo)
            puestos.append(viejo)
    for viejo, nuevo in _EXTRA.items():
        if _falta(viejo) and hasattr(numpy, nuevo):
            setattr(numpy, viejo, getattr(numpy, nuevo))
            puestos.append(viejo)
    # Para lo que numpy declare retirado y no este en la lista: su mensaje de
    # error suele nombrar el reemplazo, y de ahi se saca.
    for viejo, motivo in getattr(numpy, "__expired_attributes__", {}).items():
        nuevo = _reemplazo(motivo)
        if nuevo and _falta(viejo) and hasattr(numpy, nuevo):
            setattr(numpy, viejo, getattr(numpy, nuevo))
            puestos.append(viejo)
    return puestos


REPUESTOS = restaurar()
'''

NOMBRE_MOD = 'zzz_numpy_compat.py'
NOMBRE_PTH = 'zzz_numpy_alias_pyscenic.pth'

# El archivo tiene que quedar en la misma carpeta que numpy. Python recorre esas
# carpetas en orden y ejecuta lo que encuentra en cada una nada mas anadirla, de
# modo que colocarlo en otra haria que se ejecutase cuando numpy todavia no esta
# disponible y no serviria de nada.
_dirs = []
try:
    _dirs.append(os.path.dirname(os.path.dirname(os.path.abspath(np.__file__))))
except Exception:
    pass
if hasattr(site, 'getsitepackages'):
    _dirs.extend(d for d in site.getsitepackages() if d not in _dirs)
if hasattr(site, 'getusersitepackages'):
    _du = site.getusersitepackages()
    if _du not in _dirs:
        _dirs.append(_du)

_pth_ok, _destino = False, None
for _dir in _dirs:
    try:
        os.makedirs(_dir, exist_ok=True)
        with open(os.path.join(_dir, NOMBRE_MOD), 'w') as fh:
            fh.write(_MODULO)
        with open(os.path.join(_dir, NOMBRE_PTH), 'w') as fh:
            fh.write('import zzz_numpy_compat\n')
        _pth_ok, _destino = True, _dir
        break
    except OSError:
        continue

# Se retiran las copias que hayan quedado en otras carpetas. Se ejecutarian al
# arrancar cualquier programa del entorno y fallarian por ese mismo orden.
_limpiadas = 0
for _dir in _dirs:
    if _dir == _destino:
        continue
    for _f in (NOMBRE_PTH, NOMBRE_MOD):
        _ruta = os.path.join(_dir, _f)
        if os.path.exists(_ruta):
            try:
                os.remove(_ruta)
                _limpiadas += 1
            except OSError:
                pass

# En esta sesion se aplica el mismo archivo que usara el programa aparte, para
# que los dos se comporten igual. La instalacion ya repuso estos nombres, asi
# que lo normal es que aqui no quede nada por hacer.
if _pth_ok:
    if _destino not in sys.path:
        sys.path.insert(0, _destino)
    _mod = importlib.reload(importlib.import_module('zzz_numpy_compat'))
    restaurados = _mod.REPUESTOS
else:
    restaurados = []
    for _v, _t in (('object', object), ('bool', bool), ('int', int),
                   ('float', float), ('str', str), ('complex', complex)):
        if not hasattr(np, _v):
            setattr(np, _v, _t)
            restaurados.append(_v)

print(f"numpy {np.__version__} | alias repuestos aqui: {len(restaurados)}")
print(f"escrito en: {_destino}" + (f" | copias antiguas borradas: {_limpiadas}" if _limpiadas else ""))

# Se lanza un programa de prueba para confirmar que ahi tambien estan los
# nombres, en vez de darlo por hecho. Se comprueba uno de cada tanda de
# retiradas, porque llegaron en versiones distintas de numpy.
_r = subprocess.run([sys.executable, "-c",
                     "import numpy; print(hasattr(numpy,'object') and hasattr(numpy,'unicode_'))"],
                    capture_output=True, text=True)
_sub_ok = _r.stdout.strip() == "True"
print(f"subproceso con alias: {_sub_ok}")

if not _sub_ok:
    if _r.stderr.strip():
        print("\n--- error al arrancar el subproceso ---")
        print(_r.stderr[-1200:])
    raise RuntimeError(
        "El subproceso no tiene los alias de numpy, asi que 'pyscenic ctx' (celda\n"
        "6.4) va a fallar con AttributeError sobre np.object. Se para aqui, que es\n"
        "donde se ve la causa, en vez de dos celdas mas adelante.\n"
        f"El .pth se escribio en: {_destino}\n"
        "Revisa el error de arriba: lo habitual es que esa carpeta no sea la misma\n"
        "en la que esta instalado numpy, o que no haya permiso de escritura."
    )

### 6.1 · Descargar bases de datos de SCENIC

In [ ]:
import os, hashlib, urllib.request, urllib.error

os.makedirs('scenic_data', exist_ok=True)

# Huella digital de cada base de datos de SCENIC, para confirmar que la descarga
# llego completa y sin alterar. Si sus autores publican una version nueva la
# huella cambia y el analisis se detiene, que es lo deseable: conviene saber que
# los datos de referencia cambiaron antes de comparar resultados con corridas
# anteriores. Para adoptar la version nueva se anota aqui su huella.
CHECKSUMS = {
    'motifs.tbl':          '81eb754118e27e854974301b1400fcf519489f8be5249239671fb288cb501c31',
    # Este nombre no se puede cambiar. SCENIC deduce que tipo de base de datos
    # tiene delante a partir de la terminacion del archivo, y con cualquier otro
    # nombre se detiene nada mas empezar.
    'hg38.genes_vs_motifs.rankings.feather':
        '9c4026a3a8e25fe07cf96749644e2ca028b787410829b30b9932574dc6e78bdb',
    'expr_mat_tiny.loom':  'ca57894cc828488d7aeb3ca58ad76a637f265c502688856d45a73d39f9483b4c',
    'allTFs_hg38.txt':     None,   # sin fijar: se imprime el hash para que lo fijes tu
}

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for blq in iter(lambda: fh.read(chunk), b''):
            h.update(blq)
    return h.hexdigest()

def parece_html(path):
    """Un error 404/403 suele guardarse como pagina HTML con nombre de .feather."""
    with open(path, 'rb') as fh:
        inicio = fh.read(512).lstrip().lower()
    return inicio.startswith(b'<!doctype html') or inicio.startswith(b'<html')

def descargar_verificado(nombre, url, destino):
    esperado = CHECKSUMS.get(nombre)

    # Un archivo que ya existe puede venir cortado de un intento anterior, asi que
    # se comprueba igual antes de darlo por bueno.
    if os.path.exists(destino):
        if not VERIFICAR_CHECKSUMS or esperado is None:
            print(f"  {nombre}: {os.path.getsize(destino)/1e6:.1f} MB (ya estaba)")
            return
        obtenido = sha256(destino)
        if obtenido == esperado:
            print(f"  {nombre}: {os.path.getsize(destino)/1e6:.1f} MB  SHA256 OK")
            return
        print(f"  {nombre}: el archivo local NO coincide con el hash fijado -> se rebaja")
        os.remove(destino)

    # Se descarga con un nombre provisional y se renombra al terminar. Asi una
    # descarga interrumpida nunca queda con el nombre definitivo, haciendose
    # pasar por completa.
    parcial = destino + '.part'
    print(f"  Descargando {nombre} ...")
    try:
        with urllib.request.urlopen(url, timeout=120) as resp:
            if resp.status != 200:
                raise RuntimeError(f"HTTP {resp.status} al descargar {nombre}")
            declarado = resp.headers.get('Content-Length')
            declarado = int(declarado) if declarado else None
            with open(parcial, 'wb') as fh:
                while True:
                    trozo = resp.read(1 << 20)
                    if not trozo:
                        break
                    fh.write(trozo)
    except urllib.error.URLError as e:
        if os.path.exists(parcial):
            os.remove(parcial)
        raise RuntimeError(f"No se pudo descargar {nombre} desde {url}\n  {e}") from e

    real = os.path.getsize(parcial)
    if declarado is not None and real != declarado:
        os.remove(parcial)
        raise RuntimeError(
            f"{nombre}: descarga truncada ({real:,} de {declarado:,} bytes).\n"
            "Vuelve a ejecutar la celda; suele ser un corte de red temporal."
        )
    if parece_html(parcial):
        os.remove(parcial)
        raise RuntimeError(
            f"{nombre}: el servidor devolvio una pagina HTML, no el archivo.\n"
            f"URL probablemente caida o movida: {url}"
        )

    obtenido = sha256(parcial)
    if esperado is None:
        print(f"    (sin hash fijado) SHA256 = {obtenido}")
    elif VERIFICAR_CHECKSUMS and obtenido != esperado:
        os.remove(parcial)
        raise RuntimeError(
            f"{nombre}: SHA256 no coincide.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}\n"
            "O la descarga se corrompio (re-ejecuta), o el archivo de origen cambio.\n"
            "Si el cambio es esperado, actualiza CHECKSUMS con el hash nuevo."
        )

    os.replace(parcial, destino)
    print(f"  {nombre}: {real/1e6:.1f} MB  SHA256 {'OK' if esperado else 'registrado'}")

COM = {
 'motifs.tbl': 'https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl',
 'hg38.genes_vs_motifs.rankings.feather': 'https://resources.aertslab.org/cistarget/databases/homo_sapiens/hg38/refseq_r80/mc_v10_clust/gene_based/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather',
}
if MODO == "ejemplo":
    COM['expr_mat_tiny.loom'] = 'https://raw.githubusercontent.com/aertslab/SCENICprotocol/master/example/expr_mat_tiny.loom'

# La lista completa de factores de transcripcion humanos, que se usa en los dos
# modos. La lista reducida que acompana al protocolo de SCENIC hoy contiene un
# solo factor, y con uno solo no hay red que reconstruir. Cruzar la lista
# completa con los datos de ejemplo deja unas decenas, suficientes para ver el
# procedimiento funcionando de principio a fin.
COM['allTFs_hg38.txt'] = 'https://resources.aertslab.org/cistarget/tf_lists/allTFs_hg38.txt'

for fn, url in COM.items():
    descargar_verificado(fn, url, f'scenic_data/{fn}')

print("Bases de datos de SCENIC listas y verificadas.")


### 6.2 · Preparar matriz + TFs según el modo

In [ ]:
import pandas as pd, numpy as np

if MODO == "ejemplo":
    import loompy
    with loompy.connect('scenic_data/expr_mat_tiny.loom') as ds:
        ex_matrix = pd.DataFrame(ds[:, :].T, index=ds.ca['CellID'], columns=ds.ra['Gene'])
    all_tfs = pd.read_csv('scenic_data/allTFs_hg38.txt', header=None).iloc[:, 0].tolist()
    tf_names = [t for t in all_tfs if t in ex_matrix.columns]
    N_EST = 500
else:
    import scanpy as sc
    if 'adata_B' not in globals():          # por si se corre la seccion 6 sin la 5
        adata_B = seleccionar_celulas_B(adata, col_ctype)
    ad_s = adata_B.copy()
    if SCENIC_DOWNSAMPLE and ad_s.n_obs > SCENIC_N_CELLS:
        sc.pp.subsample(ad_s, n_obs=SCENIC_N_CELLS, random_state=0)
        print(f"Recorte a {ad_s.n_obs:,} celulas B (SCENIC_DOWNSAMPLE=True)")
    else:
        print(f"Todas las celulas B disponibles: {ad_s.n_obs:,}")
        if ad_s.n_obs > 8000:
            print("       Son bastantes: reconstruir la red puede tardar. Si hace falta")
            print("       acortarlo, pon SCENIC_DOWNSAMPLE = True en la celda 1.")

    # El pool de genes se recorta contra la base de datos de SCENIC ANTES de elegir
    # los mas variables, no despues. El orden es lo que importa aqui.
    # El filtro intermedio de SCENIC descarta cualquier grupo de genes del que no
    # reconozca al menos el 80% en esa base. Los genes de inmunoglobulina copan
    # las primeras posiciones por variabilidad y no estan en la base, asi que
    # colandose en la seleccion arrastran a casi todos los grupos por debajo del
    # umbral y el paso siguiente devuelve una lista vacia.
    # Se excluyen tambien por prefijo de nombre, ademas de por ausencia en la
    # base, porque no todas sus variantes faltan en ella.
    from ctxcore.rnkdb import FeatherRankingDatabase
    db_cistarget = FeatherRankingDatabase(
        fname="scenic_data/hg38.genes_vs_motifs.rankings.feather", name="hg38")
    genes_cistarget = set(db_cistarget.genes)
    es_ig = ad_s.var_names.str.match(r"^IG[HKL][VDJ]")
    en_cistarget = ad_s.var_names.isin(genes_cistarget)
    print(f"Genes en la matriz de celulas B: {ad_s.n_vars:,} | en la base de cisTarget: "
          f"{en_cistarget.sum():,} ({en_cistarget.mean():.0%}) | inmunoglobulinas "
          f"excluidas: {es_ig.sum():,}")
    ad_s = ad_s[:, en_cistarget & ~es_ig].copy()
    if ad_s.n_vars < 1000:
        print(f"\nAVISO: solo {ad_s.n_vars} genes quedan tras filtrar contra cisTarget. "
              "Probablemente seguira validando pocos o ningun regulon.")

    sc.pp.highly_variable_genes(ad_s, n_top_genes=min(SCENIC_N_GENES, ad_s.n_vars - 1))
    ad_s = ad_s[:, ad_s.var.highly_variable].copy()
    Xd = ad_s.X.toarray() if hasattr(ad_s.X, 'toarray') else np.asarray(ad_s.X)
    ex_matrix = pd.DataFrame(Xd, index=ad_s.obs_names.astype(str),
                             columns=ad_s.var_names.astype(str))
    all_tfs = pd.read_csv('scenic_data/allTFs_hg38.txt', header=None).iloc[:, 0].tolist()
    tf_names = [t for t in all_tfs if t in ex_matrix.columns]
    N_EST = 200

print(f"Matriz: {ex_matrix.shape[0]:,} celulas x {ex_matrix.shape[1]:,} genes | TFs: {len(tf_names)}")

# SCENIC necesita un numero suficiente de genes para detectar que secuencias de
# union estan sobrerrepresentadas. Con muy pocos, lo esperable es que no valide
# ningun regulon.
if ex_matrix.shape[1] < 1000:
    print(f"\nAVISO: solo {ex_matrix.shape[1]} genes en la matriz. cisTarget probablemente\n"
          "no validara ningun regulon (necesita suficientes targets por TF para detectar\n"
          "enriquecimiento de motivos). Es lo esperable con el dataset 'tiny' de ejemplo.")


### 6.3 · Inferencia de la red (GRNBoost2)

Primer paso de SCENIC: para cada gen se entrena un modelo que predice su expresión a partir de la de los factores de transcripción. Los TF que más ayudan a predecir un gen quedan enlazados a él. El resultado es una lista de aristas TF-gen con una importancia asociada.

Se usa `arboreto.grnboost2`, la implementación de referencia de SCENIC.



In [ ]:
import os, time, pandas as pd, numpy as np

tfs_in = [t for t in tf_names if t in ex_matrix.columns]
if not tfs_in:
    raise ValueError(
        "Ningun factor de transcripcion de la lista aparece en la matriz de expresion.\n"
        "Sin TFs no hay red que inferir. Suele significar que los nombres de gen no\n"
        "coinciden (p.ej. symbols vs Ensembl IDs)."
    )
print(f"TFs presentes en la matriz: {len(tfs_in)} de {len(tf_names)}")

# Con menos de dos entradas en la lista de reguladores no hay nada que inferir.
# Se corta aqui porque el error que llega despues, desde la libreria de calculo
# distribuido, no menciona la causa.
if len(tfs_in) < 2:
    raise ValueError(
        f"Solo hay {len(tfs_in)} TF en la matriz ({tfs_in}).\n"
        "GRNBoost2 necesita varios para tener algo que comparar. Revisa de donde\n"
        "sale la lista de TFs en la celda 6.2."
    )

t0 = time.time()

if METODO_GRN == "grnboost2":
    from arboreto.algo import grnboost2

    # La libreria que reparte el calculo entre varios procesadores no ha tenido
    # publicacion nueva desde 2020, y pide a la libreria de calculo distribuido
    # que combine una lista de resultados que en este modo siempre viene vacia.
    # Las versiones actuales rechazan esa peticion y detienen el analisis antes
    # de empezar el trabajo real.
    # Se admite ese caso devolviendo una tabla vacia, que es lo que la libreria
    # acaba descartando de todos modos. El ajuste vive solo en memoria y durante
    # esta sesion: no modifica ningun archivo instalado.
    import arboreto.core as _ac

    if getattr(_ac.from_delayed, "_parcheada_por_notebook", False):
        print("  arboreto: from_delayed ya estaba parcheado")
    else:
        _orig_from_delayed = _ac.from_delayed

        def _from_delayed_tolerante(dfs, meta=None, **kw):
            if not dfs:
                import dask.dataframe as _dd, pandas as _pd
                vacio = meta.copy() if meta is not None else _pd.DataFrame()
                return _dd.from_pandas(vacio, npartitions=1)
            return _orig_from_delayed(dfs, meta=meta, **kw)

        _from_delayed_tolerante._parcheada_por_notebook = True
        _ac.from_delayed = _from_delayed_tolerante

    def _crear_cliente():
        """Devuelve (cliente, cluster), o (None, None) si dask no arranca aqui.

        El cliente lo creamos nosotros a proposito: si se deja que arboreto monte
        el suyo, le pasa 'diagnostics_port' a distributed, que ya no lo acepta.
        """
        try:
            from distributed import Client, LocalCluster
            n_w = max(1, min(4, os.cpu_count() or 1))
            cluster = LocalCluster(n_workers=n_w, threads_per_worker=1, processes=True)
            print(f"  dask: {n_w} workers")
            return Client(cluster), cluster
        except Exception as e:
            print(f"  no se pudo crear el cliente dask ({type(e).__name__}: {e})")
            print("  se deja que arboreto use su scheduler interno")
            return None, None

    def _correr_grnboost2():
        """Ejecuta GRNBoost2. Solo se captura el fallo AL CREAR el cliente.

        Si falla grnboost2 en si, el error sube tal cual. Antes todo iba dentro
        de un unico try y cualquier excepcion se reportaba como "cliente dask no
        disponible", reintentando por otra via: un fallo de datos ("Must supply
        at least one delayed object", que en realidad era una lista de TFs con un
        solo elemento) aparecia disfrazado de problema de dask, y eso costo
        varias versiones de diagnostico equivocado.
        """
        cliente, cluster = _crear_cliente()
        try:
            return grnboost2(expression_data=ex_matrix, tf_names=tfs_in,
                             client_or_address=cliente if cliente is not None else 'local',
                             seed=42, verbose=True)
        finally:
            if cliente is not None:
                cliente.close()
                cluster.close()

    try:
        adjacencies = _correr_grnboost2()
    except Exception as e:
        raise RuntimeError(
            f"GRNBoost2 fallo: {type(e).__name__}: {e}\n\n"
            "NO bajes la version de dask para arreglar esto: ya se probo y encadena\n"
            "mas fallos todavia (ver el comentario de la celda 2.3). Lee el error de\n"
            "arriba: si habla de los datos (pocos TFs, matriz vacia), el problema esta\n"
            "en la celda 6.2, no en dask. Si dice 'Must supply at least one delayed\n"
            "object', el parche de arboreto de mas arriba no llego a aplicarse.\n"
            "Como ultimo recurso, METODO_GRN = 'sklearn_aprox' en la celda 1 es una\n"
            "aproximacion NO estandar, valida solo para demostrar el flujo.\n"
            "No se cambia de metodo automaticamente: seria cambiar el algoritmo sin que te enteres."
        ) from e

else:
    # Alternativa que sigue la misma idea, medir que reguladores predicen mejor la
    # expresion de cada gen, pero no es el metodo oficial de SCENIC: le faltan su
    # criterio de parada y su esquema de muestreo. Sirve para recorrer el flujo
    # sin depender de esa libreria, no para producir resultados definitivos.
    from sklearn.ensemble import GradientBoostingRegressor
    print("Aproximacion sklearn (NO estandar). Esto puede tardar bastante...")
    X_tfs = ex_matrix[tfs_in].values
    targets = [g for g in ex_matrix.columns if g not in tfs_in]
    registros = []
    for i, tg in enumerate(targets):
        y = ex_matrix[tg].values
        if y.std() == 0:
            continue
        gbm = GradientBoostingRegressor(n_estimators=N_EST, max_depth=3, random_state=42)
        gbm.fit(X_tfs, y)
        for tf, imp in zip(tfs_in, gbm.feature_importances_):
            if imp > 0:
                registros.append({'TF': tf, 'target': tg, 'importance': float(imp)})
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(targets)}")
    adjacencies = pd.DataFrame(registros)

if adjacencies.empty:
    raise RuntimeError("La inferencia de red no produjo ninguna arista TF-gen.")

adjacencies = adjacencies.sort_values('importance', ascending=False).reset_index(drop=True)

# La red se guarda entera. El paso siguiente aplica los umbrales propios de
# SCENIC al formar los grupos de genes, y filtrar antes se apartaria del
# procedimiento publicado.
os.makedirs('scenic_data', exist_ok=True)
adjacencies.to_csv('scenic_data/adjacencies.tsv', sep='\t', index=False)

por_tf = adjacencies.groupby('TF').size()
n_targets_posibles = ex_matrix.shape[1] - 1  # cualquier otro gen puede ser target (incl. TFs); solo se excluye a si mismo
print(f"\nMetodo: {METODO_GRN} | {time.time()-t0:.0f}s")
print(f"Aristas TF-gen: {len(adjacencies):,}")
print(f"TFs con targets: {por_tf.size} | targets por TF: "
      f"mediana {por_tf.median():.0f}, min {por_tf.min()}, max {por_tf.max()}")

# Densidad de la red resultante. Si cada regulador queda conectado a casi todos
# los genes, las puntuaciones no estan separando nada y el filtro del paso
# siguiente recortaria sobre una lista sin orden util.
if n_targets_posibles > 0:
    densidad = por_tf.median() / n_targets_posibles
    print(f"Densidad (mediana targets/TF entre genes disponibles): {densidad:.1%}")
    if densidad > 0.9:
        print(
            "\nAVISO: la red es casi un grafo completo. Las importancias apenas distinguen\n"
            "unos targets de otros, asi que los modulos que construya cisTarget seran\n"
            "practicamente arbitrarios. Con METODO_GRN='sklearn_aprox' esto es lo esperable."
        )


### 6.4 · cisTarget

In [ ]:
import loompy, numpy as np, pandas as pd, subprocess, os

LOOM = 'scenic_data/expr_mat_tiny.loom' if MODO == "ejemplo" else 'scenic_data/expr_bcells.loom'
if MODO == "zenodo":
    if os.path.exists(LOOM):
        os.remove(LOOM)          # loompy.create falla si el archivo ya existe
    loompy.create(LOOM, ex_matrix.T.values,
                  {'Gene': np.array(ex_matrix.columns)},
                  {'CellID': np.array(ex_matrix.index)})

cmd = ['pyscenic', 'ctx', 'scenic_data/adjacencies.tsv', 'scenic_data/hg38.genes_vs_motifs.rankings.feather',
       '--annotations_fname', 'scenic_data/motifs.tbl',
       '--expression_mtx_fname', LOOM,
       '--output', 'scenic_data/regulons.csv',
       '--num_workers', '2']
print("Ejecutando:", " ".join(cmd), "\n")
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.stdout.strip():
    print(proc.stdout[-2500:])
if proc.stderr.strip():
    print("--- log de cisTarget ---"); print(proc.stderr[-2500:])

# Primer caso: el proceso se interrumpio. Suele deberse a los nombres retirados
# de numpy que se reponen mas arriba.
if proc.returncode != 0:
    pista = ""
    if 'np.object' in proc.stderr or "has no attribute 'object'" in proc.stderr:
        pista = ("\nEl subproceso no tiene los alias de numpy. Vuelve a ejecutar la\n"
                 "celda 6.0 y mira lo que dice 'subproceso con alias': si sale\n"
                 "False, el .pth no llego a la carpeta donde esta instalado numpy.\n"
                 "NO bajes numpy a <1.24: rompe scanpy y medio entorno de Colab.")
    raise RuntimeError(f"'pyscenic ctx' fallo (codigo {proc.returncode}).{pista}")

# Segundo caso: el proceso termino bien y queda leer cuantas filas trae el
# archivo de salida. Que no falle no significa que haya encontrado algo.
try:
    df_reg = pd.read_csv('scenic_data/regulons.csv', index_col=[0, 1], header=[0, 1])
    n_regulons = len(df_reg)
except Exception:
    n_regulons = 0

print(f"\nRegulones validados por motivos: {n_regulons}")
if n_regulons == 0:
    print(
        "\ncisTarget termino sin error pero no valido ningun regulon.\n"
        "Esto NO es el bug de numpy: es que ningun TF tiene motivos de union\n"
        "enriquecidos entre sus targets. Causas tipicas:\n"
        "  - dataset demasiado pequeno (el 'tiny' de ejemplo tiene 500 genes)\n"
        "  - TFs cuyos motivos no estan en la base hg38 (p.ej. BRF1, de Pol III)\n"
        "  - nombres de gen que no casan con la base (symbols vs Ensembl)\n"
        "  - la red del paso anterior era demasiado densa o demasiado pobre\n"
        "La celda 6.5 decide que hacer con esto."
    )


### 6.5 · AUCell: actividad de cada regulón en cada célula

AUCell puntúa, célula a célula, cuán arriba están los genes de cada regulón en su ranking de expresión.

En `MODO="zenodo"`, por defecto el notebook se detiene aquí si cisTarget no validó ningún regulón. Una versión anterior seguía adelante construyendo "regulones" directamente desde las aristas del GRN, sin el filtro por motivos, que es justo lo que distingue a SCENIC de una simple red de coexpresión. No es un riesgo teórico: en una ejecución real guardada en `runs/`, cisTarget devolvió 0 regulones, ese atajo se activó, y el `auc_matrix.csv` resultante contenía 20 "regulones" cuyos AUC iban todos de 0,0136 a 0,0414, puro ruido. El "top 10 por actividad" separaba 0,0242 de 0,0239, diferencias en la cuarta decimal presentadas como un ranking, y sobre eso se calcularon un UMAP y 9 clusters que parecían resultados. Si aun así se quiere ver el flujo completo sin datos válidos en zenodo, se pone `PERMITIR_REGULONES_SIN_VALIDAR = True` en la celda 1.

En `MODO="ejemplo"` el notebook continúa siempre, automáticamente, sin que se tenga que tocar nada: el dataset, 500 genes, nunca va a validar regulones por motivos, así que exigir esa validación ahí impediría mostrar el mecanismo completo de la demostración. La salida queda marcada `_SIN_VALIDAR`, y el notebook lo repite en pantalla. No es un resultado biológico real, es una ilustración de cómo se ve el flujo hasta el final.


In [ ]:
from pyscenic.aucell import aucell as pyscenic_aucell
from ctxcore.genesig import GeneSignature
import pandas as pd

signatures = []

# Los datos de ejemplo tienen 500 genes, demasiado pocos para que ningun regulon
# supere la validacion. Es una consecuencia de su tamano, no un fallo, asi que
# ese modo continua siempre: existe para mostrar el procedimiento entero. En el
# estudio real la exigencia se mantiene.
continuar_sin_validar = PERMITIR_REGULONES_SIN_VALIDAR or (MODO == "ejemplo")

if n_regulons > 0:
    # Camino normal, con regulones que superaron la validacion.
    REGULONES_VALIDADOS = True

    # El archivo de salida se lee con las funciones del propio SCENIC en vez de
    # interpretarlo a mano. Su formato tiene dos particularidades que solo ellas
    # resuelven.
    # La primera es como quedan escritos los numeros dentro de las celdas de
    # texto: las versiones recientes de numpy los serializan con una notacion que
    # un lector generico no sabe interpretar.
    # La segunda es que una misma clave puede ocupar varias filas, y esas
    # funciones las agrupan en una sola entrada uniendo sus genes. Procesarlas
    # fila a fila dejaria entradas repetidas con el mismo nombre, y el paso
    # siguiente indexa por ese nombre y falla con nombres duplicados.
    from pyscenic.utils import load_motifs
    from pyscenic.transform import df2regulons

    df_motifs = load_motifs('scenic_data/regulons.csv')
    signatures = df2regulons(df_motifs)
    if not signatures:
        raise RuntimeError(
            f"cisTarget reporto {n_regulons} regulones pero df2regulons no genero\n"
            "ninguno. Revisa scenic_data/regulons.csv."
        )

elif not continuar_sin_validar:
    # Parada deliberada. Sin regulones validados, continuar produciria cifras con
    # aspecto de resultado que no significan nada.
    raise RuntimeError(
        "cisTarget no valido ningun regulon, asi que NO hay nada que puntuar con AUCell.\n"
        "\n"
        "El notebook para aqui a proposito. Construir los regulones desde el GRN sin el\n"
        "filtro por motivos elimina justo el paso que diferencia a SCENIC de una red de\n"
        "coexpresion: el resultado son puntuaciones casi identicas entre si (ruido) que\n"
        "luego se grafican como si fueran biologia.\n"
        "\n"
        "Estas en MODO='zenodo'. Que hacer:\n"
        "  - Revisa el diagnostico de la celda 6.3 (que los nombres de gen casen con la\n"
        "    base hg38, y que la densidad de la red no sea degenerada).\n"
        "  - Solo para demostrar el flujo: PERMITIR_REGULONES_SIN_VALIDAR = True en la\n"
        "    celda 1. Las salidas quedaran marcadas como NO VALIDADAS."
    )

else:
    # Se continua sin validacion, sea porque son los datos de ejemplo o porque se
    # pidio expresamente en la configuracion. Todo lo que salga de aqui queda
    # marcado como no validado.
    REGULONES_VALIDADOS = False
    causa = ("MODO='ejemplo': el dataset de juguete no puede validar por diseno"
             if MODO == "ejemplo" and not PERMITIR_REGULONES_SIN_VALIDAR
             else "PERMITIR_REGULONES_SIN_VALIDAR=True (opt-in explicito en zenodo)")
    print("=" * 70)
    print(f"ATENCION: regulones SIN validacion por motivos. Causa: {causa}")
    print("Esto NO es SCENIC completo. Los resultados no son interpretables como")
    print("actividad regulatoria real y no deben entregarse como tal.")
    print("=" * 70)
    for tf, grp in adjacencies.groupby('TF'):
        signatures.append(GeneSignature(name=f"{tf}(+)",
                                        gene2weight=dict(zip(grp['target'], grp['importance']))))

SUFIJO = "" if REGULONES_VALIDADOS else "_SIN_VALIDAR"
tam = [len(s.genes) for s in signatures]
print(f"\nFirmas: {len(signatures)} | genes por firma: "
      f"mediana {int(pd.Series(tam).median())}, min {min(tam)}, max {max(tam)}")

auc_matrix = pyscenic_aucell(ex_matrix, signatures, num_workers=1)
print("AUCell:", auc_matrix.shape)

# Estas puntuaciones miden cuan activo esta cada regulon en cada celula. Si todas
# salen parecidas, los regulones no estan distinguiendo unas celulas de otras y
# el resultado no dice nada.
rango = auc_matrix.values.max() - auc_matrix.values.min()
print(f"AUC: min {auc_matrix.values.min():.4f} | max {auc_matrix.values.max():.4f} "
      f"| rango {rango:.4f}")
if rango < 0.05:
    print("\nAVISO: el rango de AUC es minusculo; las firmas apenas separan unas celulas\n"
          "de otras. Cualquier cluster o ranking calculado sobre esto sera ruido.")


### 6.6 · Visualizar regulones

In [ ]:
import scanpy as sc

asc = sc.AnnData(X=auc_matrix.values,
                 obs=pd.DataFrame(index=auc_matrix.index.astype(str)),
                 var=pd.DataFrame(index=auc_matrix.columns.astype(str)))
sc.pp.neighbors(asc, random_state=42)
sc.tl.umap(asc, random_state=42)
sc.tl.leiden(asc, flavor='igraph', n_iterations=2, random_state=42)

# El titulo indica si los regulones estan validados. Las figuras acaban en
# presentaciones e informes separadas del mensaje que las acompanaba en pantalla,
# y esa advertencia tiene que viajar con la imagen.
etiqueta = "regulones validados por motivos" if REGULONES_VALIDADOS else "SIN VALIDAR: no interpretable"
sc.pl.umap(asc, color='leiden', title=f'Actividad de regulones ({MODO}): {etiqueta}')

salida = f"{PROJECT_DIR}/scenic_auc_{MODO}{SUFIJO}.csv"
auc_matrix.to_csv(salida)
print("Guardado:", salida)
if not REGULONES_VALIDADOS:
    print("\nRecuerda: el sufijo _SIN_VALIDAR indica que estos scores NO pasaron el\n"
          "filtro por motivos de cisTarget. No los entregues como resultado de SCENIC.")


### 6.7 · Procedencia: qué se ejecutó realmente

Deja constancia del método usado y de si los regulones pasaron la validación por motivos.


In [ ]:
# ============================================================
#   RESUMEN DE LO QUE SE EJECUTO
# ============================================================
# Deja constancia de con que datos, que metodo y que versiones se obtuvieron
# estos resultados, para que puedan reproducirse o citarse mas adelante.
import datetime, sys, importlib

def _ver(mod):
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return "no disponible"

apto = REGULONES_VALIDADOS and METODO_GRN == "grnboost2"

print("=" * 62)
print("  PROCEDENCIA DEL ANALISIS SCENIC")
print("=" * 62)
print(f"  Fecha              : {datetime.datetime.now():%Y-%m-%d %H:%M}")
print(f"  Modo de datos      : {MODO}")
print(f"  Inferencia de red  : {METODO_GRN}"
      f"{'  (algoritmo estandar de SCENIC)' if METODO_GRN == 'grnboost2' else '  (APROXIMACION NO ESTANDAR)'}")
print(f"  Aristas del GRN    : {len(adjacencies):,}")
print(f"  Regulones cisTarget: {n_regulons}")
print(f"  Validado x motivos : {'SI' if REGULONES_VALIDADOS else 'NO'}")
print(f"  Matriz AUCell      : {auc_matrix.shape[0]:,} celulas x {auc_matrix.shape[1]} regulones")
print(f"  Rango de AUC       : {auc_matrix.values.min():.4f} - {auc_matrix.values.max():.4f}")
print("-" * 62)
print(f"  Python {sys.version.split()[0]} | numpy {_ver('numpy')} | "
      f"scanpy {_ver('scanpy')} | pyscenic {_ver('pyscenic')}")
print("=" * 62)

if apto:
    print("  Algoritmo estandar + regulones validados.")
else:
    motivos = []
    if METODO_GRN != "grnboost2":
        motivos.append("la red no se infirio con GRNBoost2")
    if not REGULONES_VALIDADOS:
        motivos.append("los regulones no pasaron el filtro por motivos")
    print("  NO APTO PARA ENTREGA como resultado de SCENIC, porque "
          + " y ".join(motivos) + ".")
    if MODO == "ejemplo":
        print("  Esto es lo ESPERADO en MODO='ejemplo': el dataset de juguete (500 genes)")
        print("  nunca puede validar regulones por diseno. Sirve para demostrar el")
        print("  mecanismo completo, no como resultado biologico. Usa MODO='zenodo'")
        print("  para un analisis real.")
    else:
        print("  Sirve para demostrar el flujo del pipeline, no como resultado biologico.")
print("=" * 62)
